# THALAMUS AI — Intelligence Engine

An autonomous AI operating system designed for retail and e-commerce SMEs in Bangladesh. This notebook provides an end-to-end pipeline: deterministic business fact computation, grounded instruction synthesis, parameter-efficient fine-tuning (QLoRA) on a single GPU (T4/L4), and a validated inference runtime serving a Next.js workspace.

---

### Core Architectural Principles

1. **Deterministic Fact Layer (Zero Arithmetic Hallucination)**
   Language models excel at qualitative reasoning, contextual synthesis, and narrative drafting, but are unreliable calculators. In THALAMUS, the language model never performs arithmetic. Every metric—revenue deltas, stockout forecasts, gross margins, supplier SLA adherence, and customer churn—is pre-aggregated deterministically in pandas before being supplied to the model as context.

2. **Strict Grounding & Evidence Whitelisting**
   The model must ground its findings exclusively on the provided fact payload. Every referenced entity (`record_id`, `product_id`, `supplier_id`) must originate from the underlying database. Fabricated identifiers are caught and pruned by the runtime.

3. **Bangladesh Localization**
   All financial metrics use Bangladeshi Taka (`৳` / `BDT`), regional logistics map across 38 domestic district hubs, and prompts natively support English, standard Bengali, and Latin-script Banglish.

4. **Risk-Gated Agent Workforce**
   The engine powers a specialized seven-agent team (`sales-analyst`, `marketing-agent`, `inventory-agent`, `customer-success`, `finance-agent`, `policy-docs-agent`, and `automation-agent`). Recommendations carry explicit risk tiers (`low`, `medium`, `high`) and human-in-the-loop approval gates.

## 1. Environment & Runtime Configuration

Initialize dependencies for QLoRA fine-tuning (`unsloth`, `peft`, `trl`, `bitsandbytes`) and serving infrastructure (`fastapi`, `uvicorn`, `pyngrok`). Inspect the attached GPU runtime and set precision and context window parameters accordingly.

In [1]:
%%capture
# Unsloth resolves a compatible torch/transformers/trl/peft/bitsandbytes set.
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# Serving, validation and config parsing.
!pip install -q fastapi "uvicorn[standard]" pyngrok nest_asyncio jsonschema networkx pyyaml

In [ ]:
import os, sys, gc, re, json, glob, shutil, pathlib, zipfile, importlib, warnings
import numpy as np, pandas as pd, torch

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
if gpu:
    print(f"GPU: {gpu.name} | {gpu.total_memory/1e9:.1f} GB | compute {gpu.major}.{gpu.minor}")
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU.")

# T4 is compute 7.5: no bfloat16, no FlashAttention-2. Both must be off explicitly
# rather than left to autodetection, or training dies partway through an epoch.
IS_T4 = bool(gpu) and gpu.major < 8

if IS_T4:
    CFG = dict(
        model_name     = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # strongest bilingual 7B at 4-bit
        chat_template  = "qwen-2.5",
        max_seq_length = 3072,
        batch_size     = 1,
        grad_accum     = 16,      # effective batch 16
        context_budget = 2600,    # chars of FACTS per example
        use_fp16       = True,
        use_bf16       = False,
    )
else:
    CFG = dict(
        model_name     = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        chat_template  = "llama-3.1",
        max_seq_length = 4096,
        batch_size     = 4,
        grad_accum     = 4,
        context_budget = 3200,
        use_fp16       = False,
        use_bf16       = True,
    )

CFG.update(
    lora_r        = 16,
    lora_alpha    = 32,
    lora_dropout  = 0.0,       # Unsloth's fused path is exact only at dropout 0
    learning_rate = 2e-4,
    epochs        = 2,
    warmup_ratio  = 0.05,
    weight_decay  = 0.01,
    seed          = 3407,
    dataset_size  = 1400,
    output_dir    = "/content/thalamus_lora",
)
print(json.dumps(CFG, indent=2))

### 1.1 Data Ingestion & Directory Routing

Unpack the localized dataset (`THALAMUS_Demo_Dataset_BD_v1.zip`) and organize it into dedicated directories:
* `data/`: Tabular CSV datasets (products, inventory, suppliers, orders, payments, reviews).
* `knowledge/`: Graph entities and relationship networks.
* `business/`: Operational policies, approval rules, and enterprise profiles.

In [ ]:
DATA_DIR, KNOW_DIR, BIZ_DIR = "/content/data", "/content/knowledge", "/content/business"

if not glob.glob(f"{DATA_DIR}/*.csv"):
    if not glob.glob("/content/*.zip"):
        from google.colab import files
        print("Upload THALAMUS_Demo_Dataset_BD_v1.zip")
        files.upload()
    with zipfile.ZipFile(glob.glob("/content/*.zip")[0]) as z:
        z.extractall("/content/_raw")

    for d in (DATA_DIR, KNOW_DIR, BIZ_DIR):
        os.makedirs(d, exist_ok=True)

    # Route by parent folder, falling back to filename, so the layout of the
    # archive does not matter.
    for p in pathlib.Path("/content/_raw").rglob("*"):
        if not p.is_file():
            continue
        parent = p.parent.name.lower()
        if p.suffix == ".csv":
            shutil.copy(p, f"{DATA_DIR}/{p.name}")
        elif p.suffix == ".json":
            dest = KNOW_DIR if (parent == "knowledge" or p.stem in ("entities", "relationships")) else DATA_DIR
            shutil.copy(p, f"{dest}/{p.name}")
        elif p.suffix in (".yaml", ".yml"):
            dest = BIZ_DIR if parent in ("business", "agents") else BIZ_DIR
            shutil.copy(p, f"{dest}/{p.name}")

for label, d in [("data", DATA_DIR), ("knowledge", KNOW_DIR), ("business", BIZ_DIR)]:
    print(f"{label:<10}", sorted(os.listdir(d)) if os.path.isdir(d) else "missing")

## 2. Deterministic Fact Engine (`thalamus_core.py`)

The fact layer is exported as a standalone Python module shared by both offline dataset synthesis and online inference endpoints.

### Key Capabilities
* **Robust Schema Resolution**: Resolves column aliases across dataset revisions and normalizes booleans, strings, and missing values safely.
* **Accurate Financial Accounting**: Accounts for line-item totals (`price_bdt = unit_price × quantity`), ensuring accurate revenue, COGS, and margin calculations.
* **Materiality Guardrails**: Enforces statistical thresholds on causal claims. If revenue, late delivery, or review variances are statistically insignificant, the engine explicitly reports stability rather than attributing fictitious causes.

In [ ]:
%%writefile /content/thalamus_core.py
"""
THALAMUS fact layer -- Bangladesh edition.

Design principle unchanged: the LLM never computes a number. Every figure is
computed here in pandas and injected into the prompt as context. The model
routes, selects and narrates.

What changed for the BD dataset:
  * Currency is BDT (Taka); all money formats with the Taka sign.
  * order_items carries `quantity` and a LINE-TOTAL `price_bdt`, verified equal
    to products.unit_price_bdt * quantity. Revenue must not multiply by quantity
    again; units must sum quantity rather than count rows.
  * reviews.csv has no date column, so review timing is joined through orders.
  * Seven agents, snake_case internally and kebab-case on the wire.
  * Every narrated cause is gated on materiality. See MATERIALITY below.
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

import networkx as nx
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# Agents. Internal ids are snake_case (valid identifiers, matches the YAML in
# the dataset). The frontend and routing YAML use kebab-case. Both are accepted.
# --------------------------------------------------------------------------
AGENTS: List[str] = [
    "sales_analyst",
    "marketing_agent",
    "inventory_agent",
    "customer_success",
    "finance_agent",
    "policy_docs_agent",
    "automation_agent",
]

AGENT_LABELS = {
    "sales_analyst": "Sales Analyst",
    "marketing_agent": "Marketing Agent",
    "inventory_agent": "Inventory Agent",
    "customer_success": "Customer Success",
    "finance_agent": "Finance Agent",
    "policy_docs_agent": "Policy & Docs",
    "automation_agent": "Automation Agent",
}

_SYNONYMS = {
    "policy_docs": "policy_docs_agent", "policy_and_docs": "policy_docs_agent",
    "policy": "policy_docs_agent", "docs": "policy_docs_agent",
    "policy_doc_agent": "policy_docs_agent",
    "sales": "sales_analyst", "sales_agent": "sales_analyst",
    "sales_analyst_agent": "sales_analyst",
    "marketing": "marketing_agent", "inventory": "inventory_agent",
    "customer_success_agent": "customer_success", "customer": "customer_success",
    "finance": "finance_agent", "automation": "automation_agent",
}


def to_kebab(agent_id: str) -> str:
    return str(agent_id).strip().replace("_", "-")


def normalize_agent(agent_id: Optional[str]) -> Optional[str]:
    """Accept kebab-case, snake_case, spaces or display names; return snake_case.

    The frontend calls /api/agents/inventory-agent/run while the dataset YAML
    says inventory_agent. Rejecting either is a 404 the user cannot debug, so
    both resolve to the same agent.
    """
    if not agent_id:
        return None
    key = re.sub(r"[\s\-]+", "_", str(agent_id).strip().lower())
    if key in AGENTS:
        return key
    return _SYNONYMS.get(key)


# --------------------------------------------------------------------------
# Schema contract. BD column names carry a _bdt suffix; aliases keep the rest of
# the code currency-agnostic.
# --------------------------------------------------------------------------
COLUMN_ALIASES: Dict[str, Dict[str, List[str]]] = {
    "orders": {
        "order_id": ["order_id"],
        "customer_id": ["customer_id"],
        "status": ["order_status", "status"],
        "purchased_at": ["order_purchase_timestamp", "purchase_timestamp", "order_date"],
        "approved_at": ["order_approved_at"],
        "delivered_at": ["order_delivered_customer_date", "delivered_customer_date"],
        "estimated_at": ["order_estimated_delivery_date", "estimated_delivery_date"],
        "late": ["late_delivery", "is_late"],
    },
    "order_items": {
        "order_id": ["order_id"],
        "item_seq": ["order_item_id", "item_id"],
        "product_id": ["product_id"],
        "seller_id": ["seller_id"],
        "quantity": ["quantity", "qty"],
        "price": ["price_bdt", "price", "item_price", "line_total_bdt"],
        "freight": ["freight_value_bdt", "freight_value", "shipping_value", "freight"],
    },
    "products": {
        "product_id": ["product_id"],
        "product_name": ["product_name", "name"],
        "category": ["category", "product_category_name", "product_category"],
        "unit_price": ["unit_price_bdt", "unit_price"],
    },
    "inventory": {
        "product_id": ["product_id"],
        "supplier_id": ["supplier_id"],
        "current_stock": ["current_stock", "stock_on_hand"],
        "reorder_point": ["reorder_point"],
        "adu": ["average_daily_units", "avg_daily_units", "daily_velocity"],
        "lead_time_days": ["lead_time_days", "lead_time"],
        "unit_cost": ["unit_cost_bdt", "unit_cost", "unit_cost_brl"],
        "stock_status": ["stock_status"],
    },
    "suppliers": {
        "supplier_id": ["supplier_id"],
        "name": ["supplier_name", "name"],
        "contract_lead_time": ["lead_time_days", "contract_lead_time_days"],
        "reliability": ["reliability_score", "reliability"],
    },
    "sellers": {
        "seller_id": ["seller_id"],
        "seller_city": ["seller_city", "city"],
    },
    "customers": {
        "customer_id": ["customer_id"],
        "customer_city": ["customer_city", "city"],
        "segment": ["customer_segment", "segment"],
    },
    "reviews": {
        "review_id": ["review_id"],
        "order_id": ["order_id"],
        "score": ["review_score", "score"],
        "comment": ["review_comment", "review_comment_message", "comment"],
        "created_at": ["review_creation_date", "review_date"],
    },
    "payments": {
        "order_id": ["order_id"],
        "value": ["payment_value_bdt", "payment_value", "value"],
        "type": ["payment_type"],
        "installments": ["payment_installments", "installments"],
    },
}

CANCELLED_STATES = {"canceled", "cancelled", "unavailable"}
SAFETY_DAYS = 7
CURRENCY = "BDT"
CURRENCY_SIGN = "\u09f3"  # Taka sign

# --------------------------------------------------------------------------
# Materiality thresholds.
#
# This block is the difference between analysis and horoscope. On this dataset
# the month-over-month revenue move is -0.8%, the late-delivery rate is flat
# near 7.5%, and the correlation between lateness and review score is -0.007.
# A template that always narrates "late deliveries drove the decline" would be
# asserting a cause that is not in the data. Every contributing factor is
# emitted only if it clears one of these bars; otherwise the finding states the
# metric was stable, which is the truthful answer.
# --------------------------------------------------------------------------
MATERIALITY = {
    "revenue_delta_pct": 2.0,
    "late_rate_change_pp": 2.0,
    "aov_change_pct": 2.0,
    "correlation": 0.15,
    "score_gap": 0.25,
    "thin_margin_pct": 0.15,
    "sku_share_of_delta": 0.10,
    "supplier_spread_pp": 3.0,
}


def bdt(x: Optional[float]) -> str:
    """Format money as Taka."""
    if x is None:
        return "n/a"
    try:
        v = float(x)
    except (TypeError, ValueError):
        return "n/a"
    if np.isnan(v) or np.isinf(v):
        return "n/a"
    return f"{CURRENCY_SIGN}{v:,.0f}"


def pct(x: Optional[float], digits: int = 1) -> str:
    if x is None:
        return "n/a"
    return f"{x:+.{digits}f}%"


def parse_boolean_series(s) -> pd.Series:
    """Coerce a boolean-ish column to 0/1 regardless of how the CSV encoded it.

    orders.late_delivery arrives as Python bools when pandas sniffs the column,
    but as the strings "True"/"False" whenever the values are quoted, the file
    is read with dtype=str, or it is regenerated by a different writer. In the
    string case pd.to_numeric silently yields NaN -> 0, which zeroes the
    late-delivery rate, the supplier ranking and the review correlation without
    raising anything. Parse explicitly so the encoding cannot change the answer.
    """
    if s is None:
        return pd.Series(dtype="int64")
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    if pd.api.types.is_bool_dtype(s):
        return s.astype("int64")
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype("int64")
    truthy = {"true", "1", "1.0", "t", "yes", "y"}
    return s.astype(str).str.strip().str.lower().isin(truthy).astype("int64")


def _resolve(df: pd.DataFrame, table: str) -> pd.DataFrame:
    mapping, missing = {}, []
    for logical, candidates in COLUMN_ALIASES.get(table, {}).items():
        hit = next((c for c in candidates if c in df.columns), None)
        if hit:
            mapping[hit] = logical
        else:
            missing.append(logical)
    out = df.rename(columns=mapping)
    out.attrs["missing_columns"] = missing
    out.attrs["source_columns"] = list(df.columns)
    return out


DEFAULT_POLICIES = {
    "discount": {"maximum_percent_without_approval": 20},
    "refunds": {"maximum_bdt_without_approval": 500},
    "inventory": {"restock_recommendation_requires_approval": True,
                  "purchase_order_requires_approval": True},
    "customer_communication": {"draft_allowed_automatically": True,
                               "send_requires_approval": True},
    "finance": {"high_value_action_threshold_bdt": 5000,
                "high_value_action_requires_approval": True},
    "automation": {"enabled": True, "destructive_actions_require_approval": True},
}


def load_all(data_dir: str, knowledge_dir: Optional[str] = None,
             business_dir: Optional[str] = None) -> Dict[str, Any]:
    """Load CSVs, the knowledge graph, and business policy/profile YAML."""
    tables: Dict[str, pd.DataFrame] = {}
    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".csv"):
            continue
        tables[fname[:-4]] = _resolve(pd.read_csv(os.path.join(data_dir, fname)), fname[:-4])

    if "orders" in tables:
        o = tables["orders"]
        for col in ("purchased_at", "approved_at", "delivered_at", "estimated_at"):
            if col in o.columns:
                o[col] = pd.to_datetime(o[col], errors="coerce")
        if "late" in o.columns:
            o["late"] = parse_boolean_series(o["late"])
        elif {"delivered_at", "estimated_at"} <= set(o.columns):
            o["late"] = (o["delivered_at"] > o["estimated_at"]).astype("int64")
        else:
            o["late"] = 0
    if "reviews" in tables and "created_at" in tables["reviews"].columns:
        tables["reviews"]["created_at"] = pd.to_datetime(
            tables["reviews"]["created_at"], errors="coerce")

    graph = nx.DiGraph()
    entities, relationships = [], []
    if knowledge_dir and os.path.isdir(knowledge_dir):
        ep = os.path.join(knowledge_dir, "entities.json")
        rp = os.path.join(knowledge_dir, "relationships.json")
        if os.path.exists(ep):
            entities = json.load(open(ep, encoding="utf-8"))
            for e in entities:
                nid = e.get("entity_id") or e.get("id") or e.get("node_id")
                if nid:
                    graph.add_node(nid, type=e.get("entity_type") or e.get("type"),
                                   name=e.get("name"))
        if os.path.exists(rp):
            relationships = json.load(open(rp, encoding="utf-8"))
            for r in relationships:
                s = r.get("source") or r.get("from") or r.get("src")
                t = r.get("target") or r.get("to") or r.get("dst")
                if s and t:
                    graph.add_edge(s, t, type=r.get("relation") or r.get("type") or "related")

    policies, profile, goals = dict(DEFAULT_POLICIES), {}, {}
    if business_dir and os.path.isdir(business_dir):
        try:
            import yaml

            def _y(fn):
                p = os.path.join(business_dir, fn)
                return yaml.safe_load(open(p, encoding="utf-8")) if os.path.exists(p) else {}

            policies = (_y("business_policies.yaml") or {}).get("policies") or policies
            profile = (_y("business_profile.yaml") or {}).get("business") or {}
            goals = (_y("business_goals.yaml") or {}).get("goals") or {}
        except ImportError:
            pass

    return {"tables": tables, "graph": graph, "entities": entities,
            "relationships": relationships, "policies": policies,
            "profile": profile, "goals": goals}


# --------------------------------------------------------------------------
@dataclass
class FactStore:
    tables: Dict[str, pd.DataFrame]
    graph: nx.DiGraph = field(default_factory=nx.DiGraph)
    policies: Dict[str, Any] = field(default_factory=lambda: dict(DEFAULT_POLICIES))
    profile: Dict[str, Any] = field(default_factory=dict)
    facts: Dict[str, Any] = field(default_factory=dict)

    @property
    def orders(self) -> pd.DataFrame:
        return self.tables["orders"]

    @property
    def items(self) -> pd.DataFrame:
        return self.tables["order_items"]

    def _lines(self) -> pd.DataFrame:
        """order_items with numeric coercion applied once."""
        it = self.items.copy()
        it["qty"] = pd.to_numeric(it.get("quantity", 1), errors="coerce").fillna(1)
        it["price_n"] = pd.to_numeric(it["price"], errors="coerce").fillna(0.0)
        it["freight_n"] = pd.to_numeric(it.get("freight", 0), errors="coerce").fillna(0.0)
        it["line_revenue"] = it["price_n"] + it["freight_n"]
        return it

    def _order_revenue(self) -> pd.DataFrame:
        """One row per order: revenue, units, month, status, lateness.

        price is a LINE TOTAL in this schema (price_bdt equals
        products.unit_price_bdt * quantity to the cent), so revenue must NOT be
        multiplied by quantity again. Units must sum quantity, not count rows.
        """
        it = self._lines()
        per_order = it.groupby("order_id", as_index=False).agg(
            revenue=("line_revenue", "sum"), units=("qty", "sum"),
            lines=("product_id", "count"))
        o = self.orders.merge(per_order, on="order_id", how="left")
        o["revenue"] = o["revenue"].fillna(0.0)
        o["units"] = o["units"].fillna(0.0)
        o["month"] = o["purchased_at"].dt.to_period("M")
        status = o["status"].astype(str).str.lower()
        o["is_cancelled"] = status.isin(CANCELLED_STATES)
        o["is_delivered"] = status.eq("delivered")
        o["late"] = parse_boolean_series(o["late"]) if "late" in o.columns else 0
        return o

    def _complete_months(self, monthly: pd.DataFrame) -> pd.DataFrame:
        if len(monthly) >= 3:
            med = monthly["orders"].iloc[:-1].median()
            if monthly["orders"].iloc[-1] < 0.5 * med:
                return monthly.iloc[:-1]
        return monthly

    @property
    def anchor_month(self):
        if "_anchor_month" not in self.facts:
            self.build_revenue()
        return self.facts["_anchor_month"]

    # ---- 1. revenue -------------------------------------------------------
    def build_revenue(self) -> Dict[str, Any]:
        o = self._order_revenue()
        live = o[~o.is_cancelled]

        monthly = (live.groupby("month")
                   .agg(revenue=("revenue", "sum"), orders=("order_id", "nunique"),
                        units=("units", "sum")).reset_index())
        # Late rate belongs to delivered orders only; including shipped and
        # processing orders in the denominator dilutes it toward zero.
        deliv = o[o.is_delivered]
        lr = deliv.groupby("month")["late"].agg(["mean", "size"]).reset_index()
        lr.columns = ["month", "late_rate", "delivered"]
        monthly = monthly.merge(lr, on="month", how="left")
        monthly["late_rate"] = monthly["late_rate"].fillna(0.0).round(4)
        monthly["aov"] = (monthly.revenue / monthly.orders.replace(0, np.nan)).round(2)
        monthly["revenue"] = monthly.revenue.round(2)

        complete = self._complete_months(monthly.copy())
        self.facts["_anchor_month"] = complete.iloc[-1]["month"]

        cur = complete.iloc[-1]
        prev = complete.iloc[-2] if len(complete) >= 2 else cur
        delta = float(cur.revenue - prev.revenue)
        delta_pct = round(100 * delta / prev.revenue, 2) if prev.revenue else 0.0
        vol_effect = float((cur.orders - prev.orders) * prev.aov)
        aov_effect = float(cur.orders * (cur.aov - prev.aov))

        cur_p, prev_p = cur.month, prev.month
        cur_orders = live[live.month == cur_p]

        it = self._lines().merge(o[["order_id", "month", "is_cancelled"]],
                                 on="order_id", how="left")
        it = it[~it.is_cancelled.fillna(False)]
        sku = (it[it.month.isin([cur_p, prev_p])]
               .pivot_table(index="product_id", columns="month", values="line_revenue",
                            aggfunc="sum", fill_value=0.0))
        sku.columns = [str(c) for c in sku.columns]
        c_col, p_col = str(cur_p), str(prev_p)
        for col in (c_col, p_col):
            if col not in sku.columns:
                sku[col] = 0.0
        sku["delta"] = (sku[c_col] - sku[p_col]).round(2)

        # Only cite SKUs explaining a material share of the total move.
        floor = MATERIALITY["sku_share_of_delta"] * max(abs(delta), 1.0)
        decliners = sku[sku.delta <= -floor].sort_values("delta").head(6)
        gainers = sku[sku.delta >= floor].sort_values("delta", ascending=False).head(6)

        cur_all, prev_all = o[o.month == cur_p], o[o.month == prev_p]
        material = abs(delta_pct) >= MATERIALITY["revenue_delta_pct"]

        facts = {
            "currency": CURRENCY,
            "current_month": str(cur_p),
            "previous_month": str(prev_p),
            "current_revenue_bdt": float(cur.revenue),
            "previous_revenue_bdt": float(prev.revenue),
            "delta_bdt": round(delta, 2),
            "delta_pct": delta_pct,
            "movement_is_material": bool(material),
            "direction": ("decline" if delta_pct <= -MATERIALITY["revenue_delta_pct"]
                          else "growth" if delta_pct >= MATERIALITY["revenue_delta_pct"]
                          else "flat"),
            "current_orders": int(cur.orders),
            "previous_orders": int(prev.orders),
            "current_units": int(cur.units),
            "current_aov_bdt": float(cur.aov),
            "previous_aov_bdt": float(prev.aov),
            "aov_change_pct": round(100 * (cur.aov - prev.aov) / max(prev.aov, 1), 2),
            "volume_effect_bdt": round(vol_effect, 2),
            "basket_effect_bdt": round(aov_effect, 2),
            "current_late_rate": float(cur.late_rate),
            "previous_late_rate": float(prev.late_rate),
            "late_rate_change_pp": round(100 * float(cur.late_rate - prev.late_rate), 2),
            "current_cancel_rate": round(float(cur_all.is_cancelled.mean()), 4) if len(cur_all) else 0.0,
            "previous_cancel_rate": round(float(prev_all.is_cancelled.mean()), 4) if len(prev_all) else 0.0,
            "top_decliners": [{"product_id": pid, "delta_bdt": float(r.delta),
                               "current_bdt": float(r[c_col]), "previous_bdt": float(r[p_col])}
                              for pid, r in decliners.iterrows()],
            "top_gainers": [{"product_id": pid, "delta_bdt": float(r.delta),
                             "current_bdt": float(r[c_col]), "previous_bdt": float(r[p_col])}
                            for pid, r in gainers.iterrows()],
            "monthly_series": [{"month": str(r.month), "revenue_bdt": float(r.revenue),
                                "orders": int(r.orders), "aov_bdt": float(r.aov),
                                "late_rate": float(r.late_rate)}
                               for r in complete.tail(8).itertuples()],
            "series_min_bdt": float(complete.revenue.min()),
            "series_max_bdt": float(complete.revenue.max()),
            "series_cv_pct": round(100 * float(complete.revenue.std() / complete.revenue.mean()), 2),
            "sample_order_ids_current": cur_orders.order_id.head(6).tolist(),
            "sample_late_order_ids": cur_orders[cur_orders.late == 1].order_id.head(6).tolist(),
            "records_scanned": int(len(live)),
        }
        self.facts["revenue"] = facts
        return facts

    # ---- 2. inventory -----------------------------------------------------
    def build_inventory(self) -> Dict[str, Any]:
        inv = self.tables["inventory"].copy()
        for c in ("adu", "current_stock", "lead_time_days", "reorder_point", "unit_cost"):
            if c in inv.columns:
                inv[c] = pd.to_numeric(inv[c], errors="coerce").fillna(0.0)

        safe_adu = inv["adu"].replace(0, np.nan)
        inv["days_to_stockout"] = (inv["current_stock"] / safe_adu).round(1).fillna(999.0)
        inv["cover_gap_days"] = (inv["days_to_stockout"] - inv["lead_time_days"]).round(1)
        inv["below_reorder"] = inv["current_stock"] <= inv["reorder_point"]
        inv["cover_short"] = inv["days_to_stockout"] <= inv["lead_time_days"]
        inv["at_risk"] = inv["below_reorder"] | inv["cover_short"]
        target = inv["adu"] * (inv["lead_time_days"] + SAFETY_DAYS)
        inv["reorder_qty"] = np.ceil((target - inv["current_stock"]).clip(lower=0)).astype(int)
        inv["reorder_cost_bdt"] = (inv["reorder_qty"] * inv.get("unit_cost", 0)).round(2)

        o = self._order_revenue()
        cutoff = o.purchased_at.max() - pd.Timedelta(days=90)
        recent = o[(o.purchased_at >= cutoff) & (~o.is_cancelled)][["order_id"]]
        it = self._lines().merge(recent, on="order_id", how="inner")
        rev90 = it.groupby("product_id", as_index=False)["line_revenue"].sum()
        rev90.columns = ["product_id", "revenue_90d_bdt"]
        inv = inv.merge(rev90, on="product_id", how="left")
        inv["revenue_90d_bdt"] = inv["revenue_90d_bdt"].fillna(0).round(2)
        inv["daily_revenue_at_risk_bdt"] = (inv["revenue_90d_bdt"] / 90).round(2)

        prod = self.tables.get("products")
        if prod is not None:
            keep = [c for c in ("product_id", "category", "product_name") if c in prod.columns]
            inv = inv.merge(prod[keep], on="product_id", how="left")

        at_risk = inv[inv.at_risk].sort_values(["days_to_stockout", "revenue_90d_bdt"],
                                               ascending=[True, False])

        def row(r):
            return {"product_id": r.product_id,
                    "product_name": getattr(r, "product_name", None),
                    "supplier_id": getattr(r, "supplier_id", None),
                    "category": getattr(r, "category", None),
                    "current_stock": int(r.current_stock),
                    "reorder_point": int(r.reorder_point),
                    "average_daily_units": float(r.adu),
                    "lead_time_days": int(r.lead_time_days),
                    "days_to_stockout": float(r.days_to_stockout),
                    "cover_gap_days": float(r.cover_gap_days),
                    "reorder_qty": int(r.reorder_qty),
                    "reorder_cost_bdt": float(r.reorder_cost_bdt),
                    "daily_revenue_at_risk_bdt": float(r.daily_revenue_at_risk_bdt)}

        shipped_flag = None
        if "stock_status" in inv.columns:
            shipped_flag = int((inv.stock_status.astype(str).str.lower() == "at_risk").sum())

        facts = {
            "currency": CURRENCY,
            "total_skus": int(len(inv)),
            "at_risk_count": int(len(at_risk)),
            "at_risk_flagged_in_source": shipped_flag,
            "below_reorder_count": int(inv.below_reorder.sum()),
            "cover_short_count": int(inv.cover_short.sum()),
            "healthy_count": int(len(inv) - len(at_risk)),
            "stockout_now_count": int((inv.current_stock <= 0).sum()),
            "total_reorder_cost_bdt": round(float(at_risk.reorder_cost_bdt.sum()), 2),
            "daily_revenue_at_risk_bdt": round(float(at_risk.daily_revenue_at_risk_bdt.sum()), 2),
            "median_days_to_stockout": float(inv.days_to_stockout.median()),
            "median_lead_time_days": float(inv.lead_time_days.median()),
            "critical": [row(r) for r in at_risk.head(10).itertuples()],
            "healthy_sample": [r.product_id for r in inv[~inv.at_risk].head(5).itertuples()],
            "records_scanned": int(len(inv)),
        }
        self.facts["inventory"] = facts
        self.tables["_inventory_enriched"] = inv
        return facts

    # ---- 3. suppliers -----------------------------------------------------
    def build_suppliers(self) -> Dict[str, Any]:
        o = self._order_revenue()
        inv = self.tables["inventory"][["product_id", "supplier_id"]]
        it = self._lines().merge(inv, on="product_id", how="left")
        it = it.merge(o[["order_id", "late", "delivered_at", "estimated_at",
                         "is_cancelled", "is_delivered", "month"]], on="order_id", how="left")
        it = it[(~it.is_cancelled.fillna(False)) & (it.is_delivered.fillna(False))]
        it["delay_days"] = (it["delivered_at"] - it["estimated_at"]).dt.total_seconds() / 86400

        g = it.groupby("supplier_id").agg(
            orders=("order_id", "nunique"), skus=("product_id", "nunique"),
            late_orders=("late", "sum"), worst_delay_days=("delay_days", "max"),
            revenue_bdt=("line_revenue", "sum")).reset_index()
        g["late_rate"] = (g.late_orders / g.orders.replace(0, np.nan)).round(4)
        g["on_time_rate"] = (1 - g.late_rate).round(4)
        g["revenue_bdt"] = g.revenue_bdt.round(2)
        g["worst_delay_days"] = g.worst_delay_days.round(1)

        # Averaging every delivery lets early arrivals cancel out the late ones,
        # so a supplier with a 30% late rate can show a negative average delay.
        # Measure slippage only across orders that actually ran late.
        late_only = (it[it.late == 1].groupby("supplier_id")["delay_days"].mean()
                     .reset_index().rename(columns={"delay_days": "avg_delay_days"}))
        g = g.merge(late_only, on="supplier_id", how="left")
        g["avg_delay_days"] = g.avg_delay_days.round(2)

        sup_tbl = self.tables.get("suppliers")
        if sup_tbl is not None:
            keep = [c for c in ("supplier_id", "name", "reliability", "contract_lead_time")
                    if c in sup_tbl.columns]
            g = g.merge(sup_tbl[keep], on="supplier_id", how="left")

        ranked = g[g.orders >= 20].sort_values("late_rate", ascending=False)
        if ranked.empty:
            ranked = g.sort_values("late_rate", ascending=False)

        fleet = round(float(it["late"].mean()), 4)
        spread_pp = round(100 * float(ranked.late_rate.max() - fleet), 2) if len(ranked) else 0.0

        def row(r):
            rel = getattr(r, "reliability", np.nan)
            return {"supplier_id": r.supplier_id,
                    "supplier_name": getattr(r, "name", None),
                    "orders": int(r.orders), "skus": int(r.skus),
                    "late_rate": float(r.late_rate), "on_time_rate": float(r.on_time_rate),
                    "avg_delay_days_when_late": float(r.avg_delay_days) if pd.notna(r.avg_delay_days) else None,
                    "worst_delay_days": float(r.worst_delay_days) if pd.notna(r.worst_delay_days) else None,
                    "revenue_bdt": float(r.revenue_bdt),
                    "stated_reliability": float(rel) if pd.notna(rel) else None}

        facts = {
            "currency": CURRENCY,
            "supplier_count": int(g.supplier_id.nunique()),
            "fleet_late_rate": fleet,
            "late_rate_spread_pp": spread_pp,
            # If every supplier sits within a couple of points of the fleet
            # average, "worst supplier" is noise and must not be narrated as a
            # root cause.
            "spread_is_material": bool(spread_pp >= MATERIALITY["supplier_spread_pp"]),
            "worst_suppliers": [row(r) for r in ranked.head(5).itertuples()],
            "best_suppliers": [row(r) for r in ranked.tail(3).itertuples()],
            "sample_late_order_ids": it[it.late == 1].order_id.drop_duplicates().head(6).tolist(),
            "records_scanned": int(len(it)),
        }
        self.facts["suppliers"] = facts
        return facts

    # ---- 4. reviews -------------------------------------------------------
    def build_reviews(self) -> Dict[str, Any]:
        if "reviews" not in self.tables:
            self.facts["reviews"] = {}
            return {}
        o = self._order_revenue()
        # reviews.csv carries no timestamp, so review timing is inherited from
        # the order it belongs to.
        r = self.tables["reviews"].merge(
            o[["order_id", "late", "month", "customer_id", "is_delivered"]],
            on="order_id", how="left")
        r["score"] = pd.to_numeric(r["score"], errors="coerce")
        r = r.dropna(subset=["score"])
        rd = r[r.is_delivered.fillna(False)]

        late_mean = float(rd[rd.late == 1].score.mean()) if (rd.late == 1).any() else None
        ontime_mean = float(rd[rd.late == 0].score.mean()) if (rd.late == 0).any() else None
        corr = float(rd["score"].corr(rd["late"])) if rd["late"].nunique() > 1 else 0.0
        gap = (ontime_mean - late_mean) if (late_mean is not None and ontime_mean is not None) else None

        detractors = r[r.score <= 2]
        share_late = round(float(detractors.late.fillna(0).mean()), 4) if len(detractors) else 0.0

        by_month = (r.dropna(subset=["month"]).groupby("month")
                    .agg(avg_score=("score", "mean"), reviews=("score", "size"),
                         detractor_rate=("score", lambda s: float((s <= 2).mean())))
                    .reset_index().tail(6))

        prod = self.tables.get("products")
        by_prod = pd.DataFrame()
        if prod is not None:
            pi = self.items[["order_id", "product_id"]].merge(
                r[["order_id", "score"]], on="order_id", how="inner")
            by_prod = (pi.groupby("product_id")
                       .agg(avg_score=("score", "mean"), reviews=("score", "size")).reset_index())
            by_prod = by_prod[by_prod.reviews >= 15].sort_values("avg_score")
            if "product_name" in prod.columns and len(by_prod):
                by_prod = by_prod.merge(prod[["product_id", "product_name"]],
                                        on="product_id", how="left")

        facts = {
            "review_count": int(len(r)),
            "avg_score": round(float(r.score.mean()), 3),
            "detractor_rate": round(float((r.score <= 2).mean()), 4),
            "promoter_rate": round(float((r.score >= 4).mean()), 4),
            "avg_score_when_late": round(late_mean, 3) if late_mean is not None else None,
            "avg_score_when_on_time": round(ontime_mean, 3) if ontime_mean is not None else None,
            "score_gap_late_vs_ontime": round(gap, 3) if gap is not None else None,
            "late_score_correlation": round(corr, 3),
            # On this dataset r = -0.007 and the gap is ~0.03 points, so delivery
            # does NOT explain dissatisfaction. Narration must respect that.
            "delivery_explains_sentiment": bool(
                gap is not None and abs(gap) >= MATERIALITY["score_gap"]
                and abs(corr) >= MATERIALITY["correlation"]),
            "share_of_detractors_that_were_late": share_late,
            "monthly": [{"month": str(x.month), "avg_score": round(float(x.avg_score), 2),
                         "reviews": int(x.reviews),
                         "detractor_rate": round(float(x.detractor_rate), 4)}
                        for x in by_month.itertuples()],
            "worst_products": [{"product_id": x.product_id,
                                "product_name": getattr(x, "product_name", None),
                                "avg_score": round(float(x.avg_score), 2),
                                "reviews": int(x.reviews)}
                               for x in by_prod.head(5).itertuples()] if len(by_prod) else [],
            "best_products": [{"product_id": x.product_id,
                               "product_name": getattr(x, "product_name", None),
                               "avg_score": round(float(x.avg_score), 2),
                               "reviews": int(x.reviews)}
                              for x in by_prod.tail(3).itertuples()] if len(by_prod) else [],
            "sample_detractor_review_ids": detractors.review_id.head(6).tolist()
                if "review_id" in detractors.columns else [],
            "records_scanned": int(len(r)),
        }
        self.facts["reviews"] = facts
        return facts

    # ---- 5. customers -----------------------------------------------------
    def build_customers(self) -> Dict[str, Any]:
        o = self._order_revenue()
        live = o[~o.is_cancelled]
        anchor = live.purchased_at.max()
        recent_cut = anchor - pd.Timedelta(days=90)
        prior_cut = anchor - pd.Timedelta(days=180)

        recent = set(live[live.purchased_at >= recent_cut].customer_id)
        prior = set(live[(live.purchased_at >= prior_cut) &
                         (live.purchased_at < recent_cut)].customer_id)
        lapsed = prior - recent

        per_cust = live.groupby("customer_id").agg(
            orders=("order_id", "nunique"), revenue=("revenue", "sum"),
            last_order=("purchased_at", "max")).reset_index()
        per_cust["recency_days"] = (anchor - per_cust.last_order).dt.days
        per_cust["revenue"] = per_cust.revenue.round(2)

        cust = self.tables.get("customers")
        seg_mix, seg_value = {}, []
        if cust is not None and "segment" in cust.columns:
            per_cust = per_cust.merge(cust[["customer_id", "segment"]],
                                      on="customer_id", how="left")
            seg_mix = cust.segment.value_counts().to_dict()
            seg_value = (per_cust.groupby("segment")
                         .agg(customers=("customer_id", "nunique"),
                              revenue_bdt=("revenue", "sum"),
                              avg_orders=("orders", "mean")).round(2)
                         .reset_index().to_dict("records"))

        lapsed_df = per_cust[per_cust.customer_id.isin(lapsed)].sort_values(
            "revenue", ascending=False)

        facts = {
            "currency": CURRENCY,
            "customer_count": int(per_cust.customer_id.nunique()),
            "repeat_rate": round(float((per_cust.orders > 1).mean()), 4),
            "avg_orders_per_customer": round(float(per_cust.orders.mean()), 2),
            "avg_ltv_bdt": round(float(per_cust.revenue.mean()), 2),
            "active_90d": int(len(recent)),
            "lapsed_count": int(len(lapsed)),
            "lapsed_revenue_bdt": round(float(lapsed_df.revenue.sum()), 2),
            "segment_mix": seg_mix,
            "segment_value": seg_value,
            "top_lapsed": [{"customer_id": r.customer_id, "orders": int(r.orders),
                            "revenue_bdt": float(r.revenue),
                            "recency_days": int(r.recency_days),
                            "segment": getattr(r, "segment", None)}
                           for r in lapsed_df.head(8).itertuples()],
            "records_scanned": int(per_cust.customer_id.nunique()),
        }
        self.facts["customers"] = facts
        return facts

    # ---- 6. finance -------------------------------------------------------
    def build_finance(self) -> Dict[str, Any]:
        o = self._order_revenue()
        live = o[~o.is_cancelled]
        it = self._lines().merge(live[["order_id", "month"]], on="order_id", how="inner")

        cost = self.tables["inventory"][["product_id", "unit_cost"]].copy()
        cost["unit_cost"] = pd.to_numeric(cost["unit_cost"], errors="coerce").fillna(0)
        it = it.merge(cost, on="product_id", how="left")
        it["unit_cost"] = it["unit_cost"].fillna(0)
        # price is a line total, so COGS must scale by quantity too.
        it["cogs"] = it["unit_cost"] * it["qty"]
        it["gross_margin"] = it["price_n"] - it["cogs"]

        by_month = it.groupby("month").agg(
            revenue=("line_revenue", "sum"), goods=("price_n", "sum"),
            cogs=("cogs", "sum"), margin=("gross_margin", "sum"),
            freight=("freight_n", "sum")).reset_index()
        by_month["margin_pct"] = (by_month.margin / by_month.goods.replace(0, np.nan)).round(4)
        by_month["freight_ratio"] = (by_month.freight / by_month.revenue.replace(0, np.nan)).round(4)

        anchor = self.anchor_month
        if anchor in set(by_month["month"]):
            by_month = by_month[by_month["month"] <= anchor]

        by_sku = it.groupby("product_id").agg(
            revenue=("line_revenue", "sum"), goods=("price_n", "sum"),
            margin=("gross_margin", "sum"), units=("qty", "sum")).reset_index()
        by_sku["margin_pct"] = (by_sku.margin / by_sku.goods.replace(0, np.nan)).round(4)
        thin = by_sku[(by_sku.units >= 10) &
                      (by_sku.margin_pct < MATERIALITY["thin_margin_pct"])].sort_values("margin_pct")
        weakest = by_sku[by_sku.units >= 10].sort_values("margin_pct").head(5)

        cur = by_month.iloc[-1] if len(by_month) else None
        prev = by_month.iloc[-2] if len(by_month) >= 2 else cur

        pay_mix = {}
        pm = self.tables.get("payments")
        if pm is not None and {"type", "value"} <= set(pm.columns):
            v = pd.to_numeric(pm["value"], errors="coerce").fillna(0)
            total = v.sum()
            if total:
                pay_mix = (v.groupby(pm["type"]).sum() / total).round(4).to_dict()

        facts = {
            "currency": CURRENCY,
            "current_month": str(cur.month) if cur is not None else None,
            "current_revenue_bdt": round(float(cur.revenue), 2) if cur is not None else None,
            "current_cogs_bdt": round(float(cur.cogs), 2) if cur is not None else None,
            "current_gross_margin_bdt": round(float(cur.margin), 2) if cur is not None else None,
            "current_margin_pct": float(cur.margin_pct) if cur is not None else None,
            "previous_margin_pct": float(prev.margin_pct) if prev is not None else None,
            "margin_change_pp": round(100 * float(cur.margin_pct - prev.margin_pct), 2)
                if (cur is not None and prev is not None) else None,
            "current_freight_ratio": float(cur.freight_ratio) if cur is not None else None,
            "previous_freight_ratio": float(prev.freight_ratio) if prev is not None else None,
            "portfolio_margin_pct": round(float(by_sku.margin.sum() / max(by_sku.goods.sum(), 1)), 4),
            "thin_margin_count": int(len(thin)),
            "thin_margin_skus": [{"product_id": r.product_id, "margin_pct": float(r.margin_pct),
                                  "revenue_bdt": round(float(r.revenue), 2), "units": int(r.units)}
                                 for r in thin.head(6).itertuples()],
            "weakest_margin_skus": [{"product_id": r.product_id, "margin_pct": float(r.margin_pct),
                                     "revenue_bdt": round(float(r.revenue), 2), "units": int(r.units)}
                                    for r in weakest.itertuples()],
            "payment_mix": pay_mix,
            "open_reorder_commitment_bdt": self.facts.get("inventory", {}).get("total_reorder_cost_bdt"),
            "records_scanned": int(len(it)),
        }
        self.facts["finance"] = facts
        return facts

    # ---- 7. policy --------------------------------------------------------
    def build_policy(self) -> Dict[str, Any]:
        """Approval rules, SLA posture and supplier contract terms.

        Backs policy_docs_agent. Thresholds come from business_policies.yaml so
        the agent quotes the customer's actual rules rather than invented ones.
        """
        p = self.policies or DEFAULT_POLICIES
        fin_threshold = (p.get("finance") or {}).get("high_value_action_threshold_bdt", 5000)
        refund_cap = (p.get("refunds") or {}).get("maximum_bdt_without_approval", 500)
        discount_cap = (p.get("discount") or {}).get("maximum_percent_without_approval", 20)

        inv = self.facts.get("inventory", {})
        sup = self.facts.get("suppliers", {})
        rev = self.facts.get("revenue", {})
        reorder_total = inv.get("total_reorder_cost_bdt") or 0

        contracts = []
        sup_tbl = self.tables.get("suppliers")
        if sup_tbl is not None:
            cols = [c for c in ("supplier_id", "name", "contract_lead_time", "reliability")
                    if c in sup_tbl.columns]
            for r in sup_tbl[cols].head(12).itertuples():
                contracts.append({
                    "supplier_id": r.supplier_id,
                    "supplier_name": getattr(r, "name", None),
                    "contract_lead_time_days": int(getattr(r, "contract_lead_time", 0) or 0),
                    "stated_reliability": float(getattr(r, "reliability", 0) or 0)})

        # Where stated contract reliability disagrees with delivered performance.
        breaches = []
        for s in sup.get("worst_suppliers", []):
            stated, actual = s.get("stated_reliability"), s.get("on_time_rate")
            if stated is None or actual is None:
                continue
            if stated - actual > 0.05:
                breaches.append({"supplier_id": s["supplier_id"],
                                 "supplier_name": s.get("supplier_name"),
                                 "stated_reliability": stated,
                                 "actual_on_time_rate": actual,
                                 "shortfall_pp": round(100 * (stated - actual), 2)})

        facts = {
            "currency": CURRENCY,
            "business_name": (self.profile or {}).get("name"),
            "market": (self.profile or {}).get("market"),
            "discount_cap_pct_without_approval": discount_cap,
            "refund_cap_bdt_without_approval": refund_cap,
            "high_value_threshold_bdt": fin_threshold,
            "purchase_order_requires_approval": bool(
                (p.get("inventory") or {}).get("purchase_order_requires_approval", True)),
            "restock_recommendation_requires_approval": bool(
                (p.get("inventory") or {}).get("restock_recommendation_requires_approval", True)),
            "customer_message_send_requires_approval": bool(
                (p.get("customer_communication") or {}).get("send_requires_approval", True)),
            "customer_message_draft_allowed": bool(
                (p.get("customer_communication") or {}).get("draft_allowed_automatically", True)),
            "destructive_actions_require_approval": bool(
                (p.get("automation") or {}).get("destructive_actions_require_approval", True)),
            "automation_enabled": bool((p.get("automation") or {}).get("enabled", True)),
            "pending_reorder_value_bdt": reorder_total,
            "pending_reorder_exceeds_threshold": bool(reorder_total > fin_threshold),
            "threshold_multiple": round(reorder_total / max(fin_threshold, 1), 1),
            "supplier_contracts": contracts,
            "sla_breach_count": len(breaches),
            "sla_breaches": breaches,
            "monthly_revenue_context_bdt": rev.get("current_revenue_bdt"),
            "records_scanned": len(contracts) + len(breaches) + 6,
        }
        self.facts["policy"] = facts
        return facts

    def build_all(self) -> Dict[str, Any]:
        self.build_revenue()
        self.build_inventory()
        self.build_suppliers()
        self.build_reviews()
        self.build_customers()
        self.build_finance()
        self.build_policy()
        return self.facts


# --------------------------------------------------------------------------
# Retrieval
# --------------------------------------------------------------------------
INTENT_KEYWORDS = {
    "revenue": ["revenue", "sales", "decline", "drop", "growth", "trend", "aov", "turnover",
                "বিক্রি", "রাজস্ব", "আয়", "কমল", "কমেছে", "বৃদ্ধি", "সেল",
                "bikri", "rajossho"],
    "inventory": ["stock", "stockout", "restock", "reorder", "inventory", "sku", "out of stock",
                  "স্টক", "মজুদ", "রিস্টক", "পণ্য", "শেষ", "stok", "mojud"],
    "suppliers": ["supplier", "vendor", "lead time", "delivery", "late", "delay", "shipment",
                  "courier", "সাপ্লায়ার", "সরবরাহ", "ডেলিভারি", "দেরি", "বিলম্ব", "deri"],
    "reviews": ["review", "rating", "sentiment", "complaint", "satisfaction", "nps", "star",
                "unhappy", "রিভিউ", "রেটিং", "অভিযোগ", "সন্তুষ্ট", "মতামত", "খুশি"],
    "customers": ["customer", "churn", "retention", "repeat", "segment", "loyal", "lapsed",
                  "inactive", "গ্রাহক", "কাস্টমার", "ধরে রাখা", "ফিরে", "সেগমেন্ট", "নিষ্ক্রিয়"],
    "finance": ["margin", "profit", "cost", "cash", "cogs", "payment", "expense", "pricing",
                "freight", "bkash", "nagad", "মার্জিন", "লাভ", "খরচ", "মুনাফা", "নগদ", "দাম",
                "labh", "khoroch"],
    "policy": ["policy", "approval", "approve", "sla", "contract", "agreement", "refund",
               "discount", "threshold", "compliance", "sign-off", "rule", "return policy",
               "নীতি", "অনুমোদন", "চুক্তি", "ফেরত", "রিফান্ড", "ছাড়", "শর্ত", "niti"],
}

# Specific terms outweigh generic ones. "supplier" appears in questions about
# stock, delivery and contracts alike, so on its own it is a weak signal;
# "contract" or "SLA" is unambiguous. Without this weighting a question like
# "which supplier is breaching its contract" ties between suppliers and policy
# and resolves on dict order, sending a contract question to the stock agent.
STRONG_KEYWORDS = {
    "revenue": ["revenue", "turnover", "aov", "রাজস্ব", "টার্নওভার"],
    "inventory": ["stockout", "restock", "reorder", "out of stock", "স্টক", "রিস্টক", "মজুদ"],
    "suppliers": ["lead time", "late", "delay", "shipment", "দেরি", "বিলম্ব", "ডেলিভারি"],
    "reviews": ["review", "rating", "sentiment", "nps", "রিভিউ", "রেটিং"],
    "customers": ["churn", "retention", "lapsed", "segment", "চার্ন", "সেগমেন্ট"],
    "finance": ["margin", "cogs", "profit", "freight", "মার্জিন", "মুনাফা"],
    "policy": ["policy", "sla", "approval", "approve", "contract", "agreement", "refund",
               "discount", "threshold", "compliance", "sign-off",
               "নীতি", "অনুমোদন", "চুক্তি", "রিফান্ড", "ছাড়", "শর্ত"],
}

AGENT_DOMAINS = {
    "sales_analyst": ["revenue", "customers"],
    "marketing_agent": ["customers", "revenue", "reviews"],
    "inventory_agent": ["inventory", "suppliers"],
    "customer_success": ["reviews", "customers"],
    "finance_agent": ["finance", "revenue", "inventory"],
    "policy_docs_agent": ["policy", "suppliers", "finance"],
    "automation_agent": ["inventory", "policy", "suppliers"],
}

DOMAIN_TO_TABLE = {
    "revenue": "orders", "inventory": "inventory", "suppliers": "suppliers",
    "reviews": "reviews", "customers": "customers", "finance": "payments",
    "policy": "suppliers",
}

# Which agent owns a domain when it is the *primary* intent. automation_agent is
# deliberately absent: it executes purchase orders, so it is selected only on an
# explicit action verb, never by domain overlap.
PRIMARY_DOMAIN_AGENT = {
    "revenue": "sales_analyst", "inventory": "inventory_agent",
    "suppliers": "inventory_agent", "reviews": "customer_success",
    "customers": "marketing_agent", "finance": "finance_agent",
    "policy": "policy_docs_agent",
}

ACTION_MARKERS = [
    "raise", "execute", "automate", "trigger", "create po", "create purchase",
    "issue po", "run the workflow", "run workflow", "place the order", "place order",
    "তৈরি করুন", "চালু করুন", "অটোমেট", "অর্ডার দিন", "পাঠান", "কার্যকর করুন",
]

CAUSAL_NEIGHBOURS = {
    "revenue": ["inventory", "suppliers"], "inventory": ["suppliers", "revenue"],
    "suppliers": ["inventory", "reviews"], "reviews": ["suppliers", "customers"],
    "customers": ["reviews", "revenue"], "finance": ["inventory", "revenue"],
    "policy": ["finance", "inventory"],
}
CAUSAL_MARKERS = ["why", "cause", "reason", "explain", "driver", "behind", "root",
                  "কেন", "কারণ", "ব্যাখ্যা", "keno", "karon"]


def route_domains(query: str, agent: Optional[str] = None, max_domains: int = 3) -> List[str]:
    q = str(query).lower()
    scores = {d: sum(1 for k in kws if k in q) for d, kws in INTENT_KEYWORDS.items()}
    for d, kws in STRONG_KEYWORDS.items():
        scores[d] = scores.get(d, 0) + 2 * sum(1 for k in kws if k in q)
    hits = [d for d, s in sorted(scores.items(), key=lambda x: -x[1]) if s > 0]

    if any(m in q for m in CAUSAL_MARKERS) and hits:
        for n in CAUSAL_NEIGHBOURS.get(hits[0], []):
            if n not in hits:
                hits.append(n)

    agent = normalize_agent(agent)
    if agent:
        for d in AGENT_DOMAINS.get(agent, []):
            if d not in hits:
                hits.append(d)
    if not hits:
        hits = ["revenue", "inventory"]
    return hits[:max_domains]


def infer_agent(query: str) -> str:
    """Map a free-text Ask Thalamus query onto an agent by primary domain."""
    q = str(query).lower()
    if any(m in q for m in ACTION_MARKERS):
        return "automation_agent"
    for d in route_domains(query):
        if d in PRIMARY_DOMAIN_AGENT:
            return PRIMARY_DOMAIN_AGENT[d]
    return "sales_analyst"


def render_context(facts: Dict[str, Any], domains: List[str], budget_chars: int = 3000) -> str:
    slice_ = {d: facts.get(d, {}) for d in domains if facts.get(d)}
    text = json.dumps(slice_, ensure_ascii=False, separators=(",", ":"))
    if len(text) <= budget_chars:
        return text
    trimmed = {d: {k: (v[:3] if isinstance(v, list) else v) for k, v in blob.items()}
               for d, blob in slice_.items()}
    return json.dumps(trimmed, ensure_ascii=False, separators=(",", ":"))[:budget_chars]


def valid_ids(facts: Dict[str, Any], tables: Dict[str, pd.DataFrame]) -> Dict[str, set]:
    out = {}
    for table, col in [("orders", "order_id"), ("inventory", "product_id"),
                       ("products", "product_id"), ("suppliers", "supplier_id"),
                       ("customers", "customer_id"), ("reviews", "review_id"),
                       ("sellers", "seller_id"), ("order_items", "product_id"),
                       ("payments", "order_id")]:
        if table in tables and col in tables[table].columns:
            out[table] = set(tables[table][col].astype(str))
    return out

In [ ]:
sys.path.insert(0, "/content")
import thalamus_core as tc
importlib.reload(tc)

bundle = tc.load_all(DATA_DIR, KNOW_DIR, BIZ_DIR)
tables = bundle["tables"]

print("Loaded tables:")
for name, df in tables.items():
    miss = df.attrs.get("missing_columns") or []
    flag = f"   [unmapped: {', '.join(miss)}]" if miss else ""
    print(f"  {name:<14} {len(df):>6,} rows x {len(df.columns):>2} cols{flag}")

print(f"\nKnowledge graph : {bundle['graph'].number_of_nodes():,} nodes, "
      f"{bundle['graph'].number_of_edges():,} edges")
print(f"Business        : {bundle['profile'].get('name')} ({bundle['profile'].get('market')})")
print(f"Policy sections : {list(bundle['policies'].keys())}")
print(f"Agents          : {len(tc.AGENTS)} -> {[tc.to_kebab(a) for a in tc.AGENTS]}")

### 2.1 Boolean Parsing & Data Type Validation

Verify that boolean fields (such as `late_delivery`) are coerced into consistent integer binary states (`0` or `1`) across diverse CSV encodings (native booleans, quoted strings, numeric flags) to prevent silent metric drops.

In [ ]:
encodings = {
    "python bools (as shipped)": pd.Series([True, False, True, None]),
    "quoted strings":            pd.Series(["True", "False", "True", None]),
    "lowercase":                 pd.Series(["true", "false", "true", None]),
    "integers":                  pd.Series([1, 0, 1, None]),
    "yes/no":                    pd.Series(["yes", "no", "yes", None]),
    "read with dtype=str":       pd.Series(["True", "False", "True", "nan"]),
}
print(f"{'encoding':<28} {'old code':>9} {'new parser':>11}")
for name, s in encodings.items():
    old = pd.to_numeric(s, errors="coerce").fillna(0).astype(int).mean()
    new = tc.parse_boolean_series(s).mean()
    flag = "" if abs(old - new) < 1e-9 else "   <-- old code silently zeroes this"
    print(f"{name:<28} {old:>9.3f} {new:>11.3f}{flag}")

raw = pd.read_csv(f"{DATA_DIR}/orders.csv")["late_delivery"]
print(f"\nActual file: dtype={raw.dtype}, python types={set(type(x).__name__ for x in raw.dropna().head(20))}")
print(f"late rate, old code   {pd.to_numeric(raw, errors='coerce').fillna(0).mean():.4f}")
print(f"late rate, new parser {tc.parse_boolean_series(raw).mean():.4f}")

In [ ]:
fs = tc.FactStore(tables=tables, graph=bundle["graph"],
                  policies=bundle["policies"], profile=bundle["profile"])
FACTS = fs.build_all()

for domain in ["revenue", "inventory", "suppliers", "reviews", "customers", "finance", "policy"]:
    blob = FACTS.get(domain) or {}
    if not blob:
        continue
    print(f"\n=== {domain.upper()} ===")
    scalars = {k: v for k, v in blob.items() if not isinstance(v, (list, dict))}
    for k, v in list(scalars.items())[:10]:
        print(f"  {k:<38} {v}")

with open("/content/facts.json", "w") as f:
    json.dump(FACTS, f, ensure_ascii=False, default=str, indent=2)
print("\nSaved /content/facts.json")

### 2.2 Metric Reconciliation & Ground Truth Verification

Cross-check computed fact aggregates against direct dataframe queries to verify calculation accuracy before synthesizing training examples.

In [ ]:
rev, inv, sup, fin = FACTS["revenue"], FACTS["inventory"], FACTS["suppliers"], FACTS["finance"]

# Independent recompute of current-month revenue, straight from the raw frames.
o = pd.read_csv(f"{DATA_DIR}/orders.csv")
it = pd.read_csv(f"{DATA_DIR}/order_items.csv")
o["month"] = pd.to_datetime(o.order_purchase_timestamp).dt.to_period("M").astype(str)
live = o[~o.order_status.str.lower().isin(tc.CANCELLED_STATES)]
chk = it.merge(live[["order_id", "month"]], on="order_id").query("month == @rev['current_month']")
naive = float((chk.price_bdt + chk.freight_value_bdt).sum())

print(f"anchor month             {rev['current_month']}  (partial month dropped)")
print(f"fact layer revenue       {tc.bdt(rev['current_revenue_bdt'])}")
print(f"independent recompute    {tc.bdt(naive)}")
print(f"agreement                {'PASS' if abs(naive - rev['current_revenue_bdt']) < 1 else 'FAIL'}")

recon = rev["volume_effect_bdt"] + rev["basket_effect_bdt"]
tol = 0.02 * abs(rev["delta_bdt"] or 1)
print(f"delta decomposition      {recon:,.0f} vs actual {rev['delta_bdt']:,.0f} "
      f"({'PASS' if abs(recon - rev['delta_bdt']) < tol else 'CHECK'})")

# price_bdt must be a line total, not a unit price.
pr = pd.read_csv(f"{DATA_DIR}/products.csv")
m = it.merge(pr[["product_id", "unit_price_bdt"]], on="product_id")
err = ((m.price_bdt / m.unit_price_bdt) - m.quantity).abs().mean()
print(f"price_bdt = unit x qty   {'PASS' if err < 0.01 else 'FAIL'}  (mean error {err:.4f})")

print(f"\nat-risk SKUs             {inv['at_risk_count']} computed vs "
      f"{inv['at_risk_flagged_in_source']} flagged in stock_status")
print(f"reorder commitment       {tc.bdt(inv['total_reorder_cost_bdt'])}")
print(f"gross margin             {100*fin['current_margin_pct']:.1f}%")
print(f"fleet late rate          {100*sup['fleet_late_rate']:.2f}%  "
      f"(spread {sup['late_rate_spread_pp']}pp, material={sup['spread_is_material']})")

## 3. Instruction Dataset Synthesis (`thalamus_dataset.py`)

Generates structured instruction-tuning pairs across 22 query families spanning all seven agents. Every label is generated directly from the deterministic fact store.

### Generation Highlights
* **Multilingual Coverage**: Incorporates formal English, standard Bengali, and conversational Banglish.
* **Materiality Gating**: Trains the model to identify neutral/flat business periods accurately without inventing issues.
* **Strict Contract Compliance**: Enforces uniform JSON outputs containing agent assignment, scanned records, finding summaries, contributing factors, evidence IDs, and recommended actions.

In [ ]:
%%writefile /content/thalamus_dataset.py
"""
Instruction-tuning data generation for THALAMUS -- Bangladesh edition.

Every training label is CONSTRUCTED FROM THE FACT STORE, not written by hand and
not sampled from a teacher model. Every figure and every record_id in the target
JSON is therefore verifiable against the CSVs.

The important change from v1: findings branch on materiality. This dataset's
month-over-month revenue move is -0.8%, its late-delivery rate is flat, and
lateness has no relationship with review score (r = -0.007). A template that
always narrates "late deliveries drove the decline" would be teaching the model
to assert a cause that is not present. Where a signal fails its threshold in
thalamus_core.MATERIALITY, the label says the metric was stable and looks
elsewhere. Teaching the model to say "revenue was essentially flat" when it was
is the single highest-value behaviour in this whole pipeline.
"""
from __future__ import annotations

import json
import random
from typing import Any, Dict, List, Optional

from thalamus_core import (AGENT_DOMAINS, AGENTS, DOMAIN_TO_TABLE, MATERIALITY,
                           FactStore, bdt, normalize_agent, pct, render_context,
                           route_domains, to_kebab)

SYSTEM_PROMPT = (
    "You are THALAMUS, an AI operating system for retail and e-commerce SMEs in "
    "Bangladesh. All money is in Bangladeshi Taka (BDT). "
    "You answer business questions using ONLY the FACTS block supplied with the "
    "question. Never invent a number, a product_id, an order_id, a customer_id or "
    "a supplier_id that does not appear in FACTS. If FACTS shows a metric was "
    "stable, say it was stable -- do not manufacture a cause. "
    "Respond with a single JSON object and nothing else: no prose before or after, "
    "no markdown code fences, no explanation. "
    "The object must have exactly these keys: agent, analyzed, finding, "
    "contributingFactors, evidence, recommendedAction. "
    "Answer in the language the user asked in."
)

# Default posture per agent. automation and finance touch money, so they default
# high; policy_docs reports rules rather than changing them, so it defaults low.
RISK_TIER = {
    "sales_analyst": "low",
    "marketing_agent": "medium",
    "inventory_agent": "medium",
    "customer_success": "low",
    "finance_agent": "high",
    "policy_docs_agent": "low",
    "automation_agent": "medium",
}

FAMILY_AGENT = {
    "revenue_why": "sales_analyst", "revenue_trend": "sales_analyst",
    "revenue_sku": "sales_analyst",
    "inventory_at_risk": "inventory_agent", "inventory_sku": "inventory_agent",
    "inventory_budget": "inventory_agent",
    "supplier_worst": "inventory_agent", "supplier_single": "inventory_agent",
    "reviews_sentiment": "customer_success", "reviews_cause": "customer_success",
    "reviews_products": "customer_success",
    "customer_churn": "marketing_agent", "customer_segment": "marketing_agent",
    "promo_recommend": "marketing_agent",
    "finance_margin": "finance_agent", "finance_thin": "finance_agent",
    "finance_payments": "finance_agent",
    "policy_approval": "policy_docs_agent", "policy_sla": "policy_docs_agent",
    "policy_rules": "policy_docs_agent",
    "automation_po": "automation_agent", "automation_policy": "automation_agent",
}


def confidence(records: int, signal_strength: float) -> float:
    """Deterministic confidence: evidence volume x effect size, capped.

    A formula rather than a vibe, so the number means the same thing in every
    row. Otherwise the model learns to emit a plausible float with no
    relationship to the evidence behind it.
    """
    vol = min(1.0, records / 400.0)
    strength = min(1.0, abs(signal_strength))
    return round(min(0.97, 0.55 + 0.28 * vol + 0.17 * strength), 2)


def analyzed_block(facts: Dict[str, Any], domains: List[str]) -> List[Dict[str, Any]]:
    out = []
    for d in domains:
        n = (facts.get(d) or {}).get("records_scanned")
        if n:
            out.append({"domain": DOMAIN_TO_TABLE.get(d, d), "records_scanned": int(n)})
    return out or [{"domain": "orders", "records_scanned": 0}]


def _ev(table: str, ids: List[Any]) -> Dict[str, Any]:
    return {"table": table, "record_ids": [str(i) for i in ids if i is not None][:6]}


# --------------------------------------------------------------------------
# Question banks: English, Bangla, and Banglish (Latin-script Bangla, which is
# what SME owners actually type on a phone keyboard).
# --------------------------------------------------------------------------
QUESTIONS: Dict[str, Dict[str, List[str]]] = {
    "revenue_why": {
        "en": ["Why did revenue decline last month?",
               "Our sales dropped last month. What caused it?",
               "Explain the revenue movement in the most recent month.",
               "What is behind the change in turnover last month?",
               "Break down what happened to revenue.",
               "Revenue looks off. Give me the root cause."],
        "bn": ["গত মাসে বিক্রি কমল কেন?",
               "আমাদের রাজস্ব গত মাসে কেমন ছিল, কারণ কী?",
               "গত মাসের বিক্রির পরিবর্তনের কারণ ব্যাখ্যা করুন।",
               "বিক্রির এই অবস্থার পেছনে কী কারণ?",
               "Goto mashe bikri komlo keno?",
               "Amader sales er obostha kemon, karon ki?"],
    },
    "revenue_trend": {
        "en": ["How is revenue trending?",
               "Give me a summary of monthly sales performance.",
               "What does the revenue trend look like over recent months?",
               "Summarise our sales trajectory."],
        "bn": ["বিক্রির ধারা কেমন যাচ্ছে?",
               "মাসিক বিক্রির একটি সারসংক্ষেপ দিন।",
               "সাম্প্রতিক মাসগুলোতে রাজস্বের অবস্থা কী?",
               "Bikri r trend kemon jacche?"],
    },
    "revenue_sku": {
        "en": ["Which products moved revenue the most last month?",
               "Show me the SKUs gaining and losing sales.",
               "Which items changed the most month over month?",
               "What is driving the product mix right now?"],
        "bn": ["কোন পণ্যগুলোর বিক্রি সবচেয়ে বেশি বদলেছে?",
               "কোন পণ্যের বিক্রি বেড়েছে আর কোনটির কমেছে?",
               "পণ্যভিত্তিক বিক্রির পরিবর্তন দেখান।",
               "Kon product er bikri sobcheye beshi bodleche?"],
    },
    "inventory_at_risk": {
        "en": ["Which products are heading for a stockout?",
               "List products at risk of stockout with reorder recommendations.",
               "What do I need to restock urgently?",
               "Show me inventory that will run out before the supplier can deliver.",
               "Which SKUs need a purchase order this week?"],
        "bn": ["কোন পণ্যগুলোর স্টক শেষ হয়ে যাচ্ছে?",
               "কোন পণ্য এখনই রিস্টক করা দরকার?",
               "স্টক আউট হওয়ার ঝুঁকিতে থাকা পণ্যের তালিকা দিন।",
               "কোন পণ্যের মজুদ দ্রুত ফুরিয়ে যাবে?",
               "Kon product er stock shesh hoye jacche?"],
    },
    "inventory_sku": {
        "en": ["How many days of cover do I have on {pid}?",
               "When will {pid} run out of stock?",
               "Is {pid} at risk? What should I order?",
               "Give me the inventory status for {pid}."],
        "bn": ["{pid} পণ্যের স্টক আর কতদিন চলবে?",
               "{pid} কবে স্টক আউট হবে?",
               "{pid} এর মজুদ পরিস্থিতি জানান।",
               "{pid} er stock ar koto din cholbe?"],
    },
    "inventory_budget": {
        "en": ["How much will it cost to restock everything at risk?",
               "What is my total reorder commitment right now?",
               "How much working capital do I need for restocking?"],
        "bn": ["ঝুঁকিতে থাকা সব পণ্য রিস্টক করতে কত খরচ হবে?",
               "এখন মোট কত টাকার অর্ডার দিতে হবে?",
               "Restock korte total koto taka lagbe?"],
    },
    "supplier_worst": {
        "en": ["Which supplier has the worst delivery performance?",
               "Who is delivering late most often?",
               "Rank my suppliers by on-time delivery.",
               "Is any vendor causing delivery problems?"],
        "bn": ["কোন সাপ্লায়ারের ডেলিভারি সবচেয়ে বেশি দেরি হচ্ছে?",
               "কোন সরবরাহকারী সবচেয়ে বেশি দেরি করছে?",
               "সাপ্লায়ারদের সময়মতো ডেলিভারির ভিত্তিতে র‍্যাংক করুন।",
               "Kon supplier sobcheye beshi deri korche?"],
    },
    "supplier_single": {
        "en": ["How is supplier {sid} performing?",
               "Give me a scorecard for {sid}.",
               "Should I keep working with {sid}?"],
        "bn": ["{sid} সাপ্লায়ারের পারফরম্যান্স কেমন?",
               "{sid} এর ডেলিভারি রেকর্ড দেখান।",
               "{sid} er performance kemon?"],
    },
    "reviews_sentiment": {
        "en": ["How are customer reviews trending?",
               "What is customer sentiment right now?",
               "Are ratings getting worse?",
               "Summarise review sentiment for me."],
        "bn": ["গ্রাহক রিভিউয়ের অবস্থা কেমন?",
               "কাস্টমার সন্তুষ্টি কি কমছে?",
               "রেটিং কি খারাপ হচ্ছে?",
               "Customer review er obostha kemon?"],
    },
    "reviews_cause": {
        "en": ["Why are customers leaving bad reviews?",
               "What is driving negative ratings?",
               "Are late deliveries affecting our ratings?",
               "What are unhappy customers complaining about?"],
        "bn": ["গ্রাহকরা খারাপ রিভিউ দিচ্ছেন কেন?",
               "নেতিবাচক রেটিংয়ের কারণ কী?",
               "দেরিতে ডেলিভারি কি রেটিং নষ্ট করছে?",
               "Customer ra kharap review dicche keno?"],
    },
    "reviews_products": {
        "en": ["Which products receive the poorest reviews?",
               "Show me the worst rated items.",
               "Which products should I investigate for quality?"],
        "bn": ["কোন পণ্যগুলো সবচেয়ে খারাপ রিভিউ পাচ্ছে?",
               "সবচেয়ে কম রেটিং পাওয়া পণ্য দেখান।",
               "Kon product er rating sobcheye kharap?"],
    },
    "customer_churn": {
        "en": ["Which customers are we losing?",
               "Show me churn risk in my customer base.",
               "Who stopped ordering recently?",
               "Which valuable customers have gone quiet?"],
        "bn": ["কোন গ্রাহকরা আমাদের ছেড়ে যাচ্ছেন?",
               "কোন কাস্টমাররা সম্প্রতি অর্ডার বন্ধ করেছেন?",
               "চার্ন ঝুঁকিতে থাকা গ্রাহকদের দেখান।",
               "Kon customer ra order bondho korese?"],
    },
    "customer_segment": {
        "en": ["How is my customer base segmented?",
               "What is my repeat purchase rate?",
               "Give me a customer segmentation overview.",
               "Which customer segment is most valuable?"],
        "bn": ["আমার গ্রাহকদের সেগমেন্ট কেমন?",
               "রিপিট ক্রয়ের হার কত?",
               "কোন গ্রাহক সেগমেন্ট সবচেয়ে মূল্যবান?",
               "Customer segment er obostha ki?"],
    },
    "promo_recommend": {
        "en": ["Where should I spend my promotion budget?",
               "Recommend a campaign to run this month.",
               "Which segment should I target with a discount?"],
        "bn": ["প্রমোশনের বাজেট কোথায় খরচ করব?",
               "এই মাসে কোন ক্যাম্পেইন চালানো উচিত?",
               "কোন গ্রাহক গ্রুপে ছাড় দেওয়া উচিত?"],
    },
    "finance_margin": {
        "en": ["How is my gross margin trending?",
               "What is my profitability this month?",
               "Is my margin improving or shrinking?",
               "Give me a margin analysis."],
        "bn": ["আমার গ্রস মার্জিন কেমন যাচ্ছে?",
               "এই মাসে লাভের অবস্থা কী?",
               "মার্জিন কি বাড়ছে না কমছে?",
               "Amar margin kemon jacche?"],
    },
    "finance_thin": {
        "en": ["Which products are least profitable?",
               "Show me SKUs with the thinnest margins.",
               "Where am I making the least money on pricing?"],
        "bn": ["কোন পণ্যগুলোতে লাভ সবচেয়ে কম?",
               "সবচেয়ে কম মার্জিনের পণ্যগুলো দেখান।",
               "Kon product e labh sobcheye kom?"],
    },
    "finance_payments": {
        "en": ["What is my payment method mix?",
               "How are customers paying?",
               "Are freight costs under control?"],
        "bn": ["গ্রাহকরা কীভাবে পেমেন্ট করছেন?",
               "পেমেন্ট পদ্ধতির অনুপাত কী?",
               "ফ্রেইট খরচ কি নিয়ন্ত্রণে আছে?"],
    },
    "policy_approval": {
        "en": ["Does this restock need my approval?",
               "What is the approval threshold for a purchase order?",
               "Can the system act on this without me?",
               "Check the approval policy for the pending reorder."],
        "bn": ["এই রিস্টকের জন্য কি আমার অনুমোদন লাগবে?",
               "পারচেজ অর্ডারের অনুমোদনের সীমা কত?",
               "সিস্টেম কি আমাকে ছাড়াই কাজটি করতে পারবে?",
               "Ei order er jonno ki amar onumodon lagbe?"],
    },
    "policy_sla": {
        "en": ["Are any suppliers breaching their contract terms?",
               "Which supplier agreements are not being honoured?",
               "Check supplier SLA compliance.",
               "Is stated supplier reliability matching actual performance?"],
        "bn": ["কোন সাপ্লায়ার চুক্তির শর্ত ভঙ্গ করছে?",
               "সাপ্লায়ারদের SLA মানা হচ্ছে কি না যাচাই করুন।",
               "চুক্তিতে বলা নির্ভরযোগ্যতা কি বাস্তবে মিলছে?",
               "Kon supplier contract vangche?"],
    },
    "policy_rules": {
        "en": ["What is our refund policy limit?",
               "How much discount can I give without approval?",
               "Summarise our operating policies.",
               "What are the rules for sending customer messages?"],
        "bn": ["আমাদের রিফান্ড নীতির সীমা কত?",
               "অনুমোদন ছাড়া কত ছাড় দেওয়া যাবে?",
               "আমাদের পরিচালনা নীতিগুলো সংক্ষেপে বলুন।",
               "Refund policy r limit koto?"],
    },
    "automation_po": {
        "en": ["Raise purchase orders for everything at risk.",
               "Automate the restock for critical SKUs.",
               "Execute the reorder workflow.",
               "Create POs for the stockout list."],
        "bn": ["ঝুঁকিতে থাকা সব পণ্যের জন্য পারচেজ অর্ডার তৈরি করুন।",
               "জরুরি পণ্যগুলোর রিস্টক অটোমেট করুন।",
               "রিঅর্ডার ওয়ার্কফ্লো চালু করুন।"],
    },
    "automation_policy": {
        "en": ["Can you approve this restock automatically?",
               "Run the workflow but check policy first.",
               "Trigger the restock if policy allows it."],
        "bn": ["এই রিস্টক কি স্বয়ংক্রিয়ভাবে অনুমোদন করা যাবে?",
               "নীতি যাচাই করে ওয়ার্কফ্লো চালু করুন।",
               "নীতি অনুমতি দিলে রিস্টক কার্যকর করুন।"],
    },
}


# --------------------------------------------------------------------------
# Label builders. Each returns the THALAMUS JSON object, branching on the
# materiality flags computed in thalamus_core.
# --------------------------------------------------------------------------
def build_label(family: str, facts: Dict[str, Any], lang: str,
                entity: Optional[str] = None) -> Optional[Dict[str, Any]]:
    agent = FAMILY_AGENT[family]
    rev = facts.get("revenue", {})
    inv = facts.get("inventory", {})
    sup = facts.get("suppliers", {})
    rvw = facts.get("reviews", {})
    cus = facts.get("customers", {})
    fin = facts.get("finance", {})
    pol = facts.get("policy", {})
    bn = lang == "bn"

    # ---------------- sales ----------------
    if family == "revenue_why":
        if not rev:
            return None
        vol, bas = rev.get("volume_effect_bdt", 0), rev.get("basket_effect_bdt", 0)
        denom = abs(rev.get("previous_revenue_bdt") or 1)
        material = rev.get("movement_is_material")
        direction = rev.get("direction")

        factors = [
            {"factor": ("অর্ডার সংখ্যা {} থেকে {}".format(rev["previous_orders"], rev["current_orders"])
                        if bn else
                        f"Order volume {rev['previous_orders']} to {rev['current_orders']}"),
             "magnitude": f"{100 * vol / denom:+.1f}%"},
            {"factor": ("গড় অর্ডার মূল্য {} থেকে {}".format(bdt(rev["previous_aov_bdt"]), bdt(rev["current_aov_bdt"]))
                        if bn else
                        f"Average order value {bdt(rev['previous_aov_bdt'])} to {bdt(rev['current_aov_bdt'])}"),
             "magnitude": f"{100 * bas / denom:+.1f}%"},
        ]
        # Only cite delivery if it actually moved.
        if abs(rev.get("late_rate_change_pp", 0)) >= MATERIALITY["late_rate_change_pp"]:
            factors.append({
                "factor": ("দেরিতে ডেলিভারির হার পরিবর্তন" if bn else "Late delivery rate change"),
                "magnitude": f"{rev['late_rate_change_pp']:+.1f}pp"})
        dec = rev.get("top_decliners") or []
        gai = rev.get("top_gainers") or []
        if dec:
            factors.append({
                "factor": (f"{dec[0]['product_id']} এর বিক্রি কমেছে" if bn else
                           f"{dec[0]['product_id']} lost {bdt(abs(dec[0]['delta_bdt']))}"),
                "magnitude": f"{100 * dec[0]['delta_bdt'] / denom:+.1f}%"})

        if material:
            finding = (
                "{} মাসে রাজস্ব {}, আগের মাস {} এ ছিল {} — পরিবর্তন {} ({})। "
                "অর্ডার সংখ্যার প্রভাব {} এবং বাস্কেট সাইজের প্রভাব {}।"
            ).format(rev["current_month"], bdt(rev["current_revenue_bdt"]), rev["previous_month"],
                     bdt(rev["previous_revenue_bdt"]), bdt(abs(rev["delta_bdt"])),
                     pct(rev["delta_pct"]), bdt(vol), bdt(bas)) if bn else (
                f"Revenue for {rev['current_month']} was {bdt(rev['current_revenue_bdt'])} against "
                f"{bdt(rev['previous_revenue_bdt'])} in {rev['previous_month']}, a {direction} of "
                f"{bdt(abs(rev['delta_bdt']))} ({pct(rev['delta_pct'])}). Splitting the move: order "
                f"volume contributed {bdt(vol)} and basket size contributed {bdt(bas)}.")
        else:
            finding = (
                "রাজস্ব আসলে কমেনি। {} মাসে {} এবং {} মাসে {} — পার্থক্য মাত্র {}, যা স্বাভাবিক "
                "মাসিক ওঠানামার ({} শতাংশ) মধ্যেই পড়ে। অর্ডার সংখ্যা {} থেকে {} এ গেছে এবং গড় অর্ডার "
                "মূল্য {} পরিবর্তিত হয়েছে, দুটি প্রভাব প্রায় একে অপরকে বাতিল করেছে। "
                "উল্লেখযোগ্য কোনো একক কারণ এই মাসে নেই — তবে পণ্যভিত্তিক মিশ্রণ বদলেছে: {} টি SKU "
                "উল্লেখযোগ্যভাবে কমেছে এবং {} টি বেড়েছে।"
            ).format(rev["current_month"], bdt(rev["current_revenue_bdt"]), rev["previous_month"],
                     bdt(rev["previous_revenue_bdt"]), pct(rev["delta_pct"]),
                     rev.get("series_cv_pct"), rev["previous_orders"], rev["current_orders"],
                     pct(rev.get("aov_change_pct")), len(dec), len(gai)) if bn else (
                f"Revenue did not materially decline. {rev['current_month']} came in at "
                f"{bdt(rev['current_revenue_bdt'])} against {bdt(rev['previous_revenue_bdt'])} in "
                f"{rev['previous_month']} — a difference of {pct(rev['delta_pct'])}, which sits well "
                f"inside the normal month-to-month variation of this business "
                f"({rev.get('series_cv_pct')}% coefficient of variation over the trailing series). "
                f"Order count actually rose from {rev['previous_orders']} to {rev['current_orders']}, "
                f"contributing {bdt(vol)}, while average order value slipped "
                f"{pct(rev.get('aov_change_pct'))} for {bdt(bas)}; the two effects nearly cancel. "
                f"Late delivery was flat at {100*rev['current_late_rate']:.1f}% and cancellations were "
                f"unchanged, so neither explains anything here. What did change is product mix: "
                f"{len(dec)} SKUs fell materially and {len(gai)} rose, roughly offsetting.")

        evidence = [_ev("orders", rev.get("sample_order_ids_current", []))]
        if dec:
            evidence.append(_ev("order_items", [d["product_id"] for d in dec]))
        if inv.get("critical"):
            evidence.append(_ev("inventory", [c["product_id"] for c in inv["critical"]]))

        action = {
            "action": (("বিক্রি স্থিতিশীল; পণ্যভিত্তিক মিশ্রণের পরিবর্তন পর্যালোচনা করুন এবং "
                        "স্টক ঝুঁকিতে থাকা {} টি SKU আগে সামলান".format(inv.get("at_risk_count", 0)))
                       if bn else
                       (f"Treat this as a flat month, not a decline. Review the SKU mix shift "
                        f"({dec[0]['product_id'] if dec else 'top decliners'} down, "
                        f"{gai[0]['product_id'] if gai else 'top gainers'} up) and prioritise the "
                        f"{inv.get('at_risk_count', 0)} SKUs at stockout risk, which is the live "
                        f"threat to next month's revenue"))
                      if not material else
                      (("রাজস্বের পরিবর্তনের প্রধান চালক পর্যালোচনা করুন এবং ঝুঁকিপূর্ণ SKU গুলো রিস্টক করুন")
                       if bn else
                       f"Address the largest driver first and restock the "
                       f"{inv.get('at_risk_count', 0)} at-risk SKUs before they compound next month"),
            "riskTier": RISK_TIER[agent],
            "confidence": confidence(rev.get("records_scanned", 0),
                                     abs(rev.get("delta_pct", 0)) / 10 if material else 0.8),
            "requiresApproval": False,
        }
        return {"agent": agent, "analyzed": analyzed_block(facts, ["revenue", "inventory", "suppliers"]),
                "finding": finding, "contributingFactors": factors[:4],
                "evidence": evidence, "recommendedAction": action}

    if family == "revenue_trend":
        series = rev.get("monthly_series") or []
        if not series:
            return None
        first, last = series[0], series[-1]
        span = 100 * (last["revenue_bdt"] - first["revenue_bdt"]) / max(first["revenue_bdt"], 1)
        finding = (
            "{} থেকে {} পর্যন্ত রাজস্ব {} থেকে {} ({})। মাসিক ওঠানামা {} শতাংশ, সর্বনিম্ন {} এবং "
            "সর্বোচ্চ {}। গড় অর্ডার মূল্য {} এর কাছাকাছি স্থিতিশীল।"
        ).format(first["month"], last["month"], bdt(first["revenue_bdt"]), bdt(last["revenue_bdt"]),
                 pct(span), rev.get("series_cv_pct"), bdt(rev.get("series_min_bdt")),
                 bdt(rev.get("series_max_bdt")), bdt(rev["current_aov_bdt"])) if bn else (
            f"Across {first['month']} to {last['month']} revenue moved from {bdt(first['revenue_bdt'])} "
            f"to {bdt(last['revenue_bdt'])} ({pct(span)}). The series is tight: a "
            f"{rev.get('series_cv_pct')}% coefficient of variation between a low of "
            f"{bdt(rev.get('series_min_bdt'))} and a high of {bdt(rev.get('series_max_bdt'))}. "
            f"Average order value has held near {bdt(rev['current_aov_bdt'])} throughout, so the "
            f"month-to-month wobble is order count rather than basket size. This is a stable "
            f"business, not a growing or shrinking one.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["revenue"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "অর্ডার সংখ্যার পরিবর্তন" if bn else "Change in order volume",
                     "magnitude": f"{100*(rev['current_orders']-rev['previous_orders'])/max(rev['previous_orders'],1):+.1f}%"},
                    {"factor": "গড় অর্ডার মূল্যের পরিবর্তন" if bn else "Change in average order value",
                     "magnitude": f"{rev.get('aov_change_pct', 0):+.1f}%"},
                    {"factor": "মাসিক ভেদাঙ্ক" if bn else "Month-to-month variation",
                     "magnitude": f"{rev.get('series_cv_pct')}%"}],
                "evidence": [_ev("orders", rev.get("sample_order_ids_current", []))],
                "recommendedAction": {
                    "action": ("পরবর্তী মাসের লক্ষ্য গত তিন মাসের গড় অর্ডার সংখ্যার ভিত্তিতে নির্ধারণ করুন"
                               if bn else
                               "Set next month's target from the trailing three-month average rather "
                               "than the peak month; growth here needs a new lever, not better execution"),
                    "riskTier": "low",
                    "confidence": confidence(rev.get("records_scanned", 0), 0.7),
                    "requiresApproval": False}}

    if family == "revenue_sku":
        dec, gai = rev.get("top_decliners") or [], rev.get("top_gainers") or []
        if not dec and not gai:
            return None
        denom = abs(rev.get("previous_revenue_bdt") or 1)
        lost = sum(d["delta_bdt"] for d in dec)
        won = sum(g["delta_bdt"] for g in gai)
        finding = (
            "মোট রাজস্ব {} হলেও পণ্যভিত্তিক মিশ্রণ যথেষ্ট বদলেছে। {} টি SKU মিলে {} হারিয়েছে, "
            "সবচেয়ে বেশি {} ({})। অন্যদিকে {} টি SKU মিলে {} যোগ করেছে, শীর্ষে {} ({})। "
            "দুটি প্রায় সমান, তাই মোট সংখ্যায় পরিবর্তন দেখা যায় না।"
        ).format(pct(rev["delta_pct"]), len(dec), bdt(abs(lost)),
                 dec[0]["product_id"] if dec else "n/a", bdt(abs(dec[0]["delta_bdt"])) if dec else "n/a",
                 len(gai), bdt(won), gai[0]["product_id"] if gai else "n/a",
                 bdt(gai[0]["delta_bdt"]) if gai else "n/a") if bn else (
            f"Headline revenue moved only {pct(rev['delta_pct'])}, but the product mix underneath it "
            f"churned considerably. {len(dec)} SKUs lost a combined {bdt(abs(lost))}, led by "
            f"{dec[0]['product_id'] if dec else 'n/a'} at {bdt(abs(dec[0]['delta_bdt'])) if dec else 'n/a'}. "
            f"Against that, {len(gai)} SKUs added {bdt(won)}, led by "
            f"{gai[0]['product_id'] if gai else 'n/a'} at {bdt(gai[0]['delta_bdt']) if gai else 'n/a'}. "
            f"The two sides nearly cancel, which is why the total looks quiet. Mix rotation of this "
            f"size is worth understanding even when the top line does not move.")
        factors = [{"factor": d["product_id"], "magnitude": f"{100*d['delta_bdt']/denom:+.1f}%"}
                   for d in dec[:2]]
        factors += [{"factor": g["product_id"], "magnitude": f"{100*g['delta_bdt']/denom:+.1f}%"}
                    for g in gai[:2]]
        return {"agent": agent, "analyzed": analyzed_block(facts, ["revenue", "inventory"]),
                "finding": finding, "contributingFactors": factors or [{"factor": "No material SKU movement", "magnitude": "0.0%"}],
                "evidence": [_ev("order_items", [d["product_id"] for d in dec + gai]),
                             _ev("orders", rev.get("sample_order_ids_current", []))],
                "recommendedAction": {
                    "action": ("কমে যাওয়া SKU গুলোর স্টক কভার যাচাই করুন — চাহিদা কমেছে নাকি স্টক ছিল না"
                               if bn else
                               f"Check stock cover on {', '.join(d['product_id'] for d in dec[:3])} "
                               f"before assuming demand shifted; a SKU that was out of stock looks "
                               f"identical to a SKU nobody wanted"),
                    "riskTier": "low",
                    "confidence": confidence(rev.get("records_scanned", 0), 0.8),
                    "requiresApproval": False}}

    # ---------------- inventory ----------------
    if family == "inventory_at_risk":
        crit = inv.get("critical") or []
        if not crit:
            return None
        top = crit[0]
        finding = (
            "{} টি SKU এর মধ্যে {} টি ঝুঁকিতে — {} টি রিঅর্ডার পয়েন্টের নিচে এবং {} টির কভার "
            "লিড টাইমের চেয়ে কম। সবচেয়ে জরুরি {} ({}): মজুদ {} ইউনিট, দৈনিক বিক্রি {}, অর্থাৎ "
            "{} দিনে শেষ, কিন্তু সাপ্লায়ার লিড টাইম {} দিন — {} দিন স্টক শূন্য থাকবে। "
            "সম্পূর্ণ তালিকা রিস্টক করতে {} লাগবে এবং দৈনিক {} বিক্রি সুরক্ষিত হবে।"
        ).format(inv["total_skus"], inv["at_risk_count"], inv.get("below_reorder_count"),
                 inv.get("cover_short_count"), top["product_id"], top.get("product_name"),
                 top["current_stock"], top["average_daily_units"], top["days_to_stockout"],
                 top["lead_time_days"], abs(top["cover_gap_days"]),
                 bdt(inv["total_reorder_cost_bdt"]), bdt(inv["daily_revenue_at_risk_bdt"])) if bn else (
            f"{inv['at_risk_count']} of {inv['total_skus']} SKUs are at risk: "
            f"{inv.get('below_reorder_count')} sit below their reorder point and "
            f"{inv.get('cover_short_count')} have less cover than their supplier lead time. "
            f"{top['product_id']} ({top.get('product_name')}) is the most urgent — "
            f"{top['current_stock']} units against {top['average_daily_units']} sold per day is "
            f"{top['days_to_stockout']} days of cover, while the supplier needs "
            f"{top['lead_time_days']} days. That is a {abs(top['cover_gap_days'])} day hole with an "
            f"empty shelf even if the PO goes out today. Clearing the full list costs "
            f"{bdt(inv['total_reorder_cost_bdt'])} and protects "
            f"{bdt(inv['daily_revenue_at_risk_bdt'])} of daily sales.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["inventory", "suppliers"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": (f"{c['product_id']}: {c['days_to_stockout']} দিন কভার, লিড টাইম {c['lead_time_days']} দিন"
                                if bn else
                                f"{c['product_id']} has {c['days_to_stockout']}d cover against a {c['lead_time_days']}d lead time"),
                     "magnitude": f"{c['cover_gap_days']:+.0f}d"} for c in crit[:4]],
                "evidence": [_ev("inventory", [c["product_id"] for c in crit]),
                             _ev("suppliers", sorted({c["supplier_id"] for c in crit if c.get("supplier_id")}))],
                "recommendedAction": {
                    "action": ("শীর্ষ {} টি SKU এর জন্য এখনই পারচেজ অর্ডার দিন, খরচ {}".format(
                        min(3, len(crit)), bdt(sum(c["reorder_cost_bdt"] for c in crit[:3])))
                        if bn else
                        f"Raise purchase orders for {', '.join(c['product_id'] for c in crit[:3])} at "
                        f"{bdt(sum(c['reorder_cost_bdt'] for c in crit[:3]))}, and expedite freight on "
                        f"{top['product_id']} where the cover gap is widest"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(inv.get("records_scanned", 0) * 4, 0.9),
                    "requiresApproval": True}}

    if family == "inventory_sku":
        crit = {c["product_id"]: c for c in inv.get("critical", [])}
        c = crit.get(entity)
        if not c:
            return None
        finding = (
            "{} ({}) এর মজুদ {} ইউনিট, দৈনিক গড় বিক্রি {} ইউনিট, অর্থাৎ {} দিনের কভার। "
            "সাপ্লায়ার {} এর লিড টাইম {} দিন, তাই আজ অর্ডার দিলেও প্রায় {} দিন স্টক শূন্য থাকবে। "
            "রিঅর্ডার পয়েন্ট {} এবং বর্তমান মজুদ তার নিচে। এই SKU দৈনিক {} বিক্রি বহন করে।"
        ).format(c["product_id"], c.get("product_name"), c["current_stock"], c["average_daily_units"],
                 c["days_to_stockout"], c.get("supplier_id"), c["lead_time_days"],
                 abs(c["cover_gap_days"]), c["reorder_point"],
                 bdt(c["daily_revenue_at_risk_bdt"])) if bn else (
            f"{c['product_id']} ({c.get('product_name')}) has {c['current_stock']} units against "
            f"average daily sales of {c['average_daily_units']}, giving {c['days_to_stockout']} days "
            f"of cover. Supplier {c.get('supplier_id')} takes {c['lead_time_days']} days, so a PO "
            f"raised today still leaves roughly {abs(c['cover_gap_days'])} days with nothing on the "
            f"shelf. Stock is below the reorder point of {c['reorder_point']}, and this SKU carries "
            f"{bdt(c['daily_revenue_at_risk_bdt'])} of sales per day.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["inventory"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "কভার গ্যাপ" if bn else "Cover gap against supplier lead time",
                     "magnitude": f"{c['cover_gap_days']:+.0f}d"},
                    {"factor": "ঝুঁকিতে থাকা দৈনিক বিক্রি" if bn else "Daily revenue exposed",
                     "magnitude": bdt(c["daily_revenue_at_risk_bdt"])}],
                "evidence": [_ev("inventory", [c["product_id"]]),
                             _ev("suppliers", [c.get("supplier_id")])],
                "recommendedAction": {
                    "action": ("{} ইউনিট অর্ডার করুন, খরচ {}".format(c["reorder_qty"], bdt(c["reorder_cost_bdt"]))
                               if bn else
                               f"Order {c['reorder_qty']} units of {c['product_id']} at approximately "
                               f"{bdt(c['reorder_cost_bdt'])}, sized to lead time plus a seven day buffer"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(200, 0.9),
                    "requiresApproval": True}}

    if family == "inventory_budget":
        crit = inv.get("critical") or []
        if not crit:
            return None
        total = inv["total_reorder_cost_bdt"]
        monthly_rev = rev.get("current_revenue_bdt") or 1
        finding = (
            "ঝুঁকিতে থাকা {} টি SKU সম্পূর্ণ রিস্টক করতে প্রায় {} প্রয়োজন। এটি মাসিক রাজস্বের "
            "({}) প্রায় {} গুণ, তাই একসাথে দেওয়া বাস্তবসম্মত নয়। শীর্ষ ৩ টি SKU একাই {} নেয়। "
            "এই তালিকা দৈনিক {} বিক্রি সুরক্ষিত রাখে।"
        ).format(inv["at_risk_count"], bdt(total), bdt(monthly_rev), round(total / monthly_rev, 1),
                 bdt(sum(c["reorder_cost_bdt"] for c in crit[:3])),
                 bdt(inv["daily_revenue_at_risk_bdt"])) if bn else (
            f"Fully restocking the {inv['at_risk_count']} at-risk SKUs needs roughly {bdt(total)} of "
            f"purchase orders. Put that in context: it is about {total/monthly_rev:.1f}x a full month "
            f"of revenue ({bdt(monthly_rev)}), so funding it in one block is not realistic and the "
            f"list has to be staged. The top three SKUs alone account for "
            f"{bdt(sum(c['reorder_cost_bdt'] for c in crit[:3]))}. Against the outlay, the at-risk "
            f"list protects {bdt(inv['daily_revenue_at_risk_bdt'])} of sales per day.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["inventory", "finance"]),
                "finding": finding,
                "contributingFactors": [{"factor": c["product_id"], "magnitude": bdt(c["reorder_cost_bdt"])}
                                        for c in crit[:4]],
                "evidence": [_ev("inventory", [c["product_id"] for c in crit])],
                "recommendedAction": {
                    "action": ("দৈনিক বিক্রির ভিত্তিতে শীর্ষ SKU গুলো আগে অর্ডার করুন, বাকিগুলো পরের চক্রে"
                               if bn else
                               "Stage the POs: fund the SKUs with the widest cover gap and highest "
                               "daily revenue first, defer slow movers to the next cycle"),
                    "riskTier": "high",
                    "confidence": confidence(inv.get("records_scanned", 0) * 4, 0.85),
                    "requiresApproval": True}}

    # ---------------- suppliers ----------------
    if family == "supplier_worst":
        worst = sup.get("worst_suppliers") or []
        if not worst:
            return None
        w = worst[0]
        spread_material = sup.get("spread_is_material")
        if spread_material:
            finding = (
                "{} টি সাপ্লায়ারের মধ্যে {} ({}) সবচেয়ে খারাপ — {} টি অর্ডারের {:.1f}% দেরি, "
                "দেরির ক্ষেত্রে গড় বিলম্ব {} দিন। এই সাপ্লায়ারের মাধ্যমে {} রাজস্ব যায়।"
            ).format(sup["supplier_count"], w["supplier_id"], w.get("supplier_name"), w["orders"],
                     100 * w["late_rate"], w.get("avg_delay_days_when_late"),
                     bdt(w["revenue_bdt"])) if bn else (
                f"Of {sup['supplier_count']} suppliers, {w['supplier_id']} ({w.get('supplier_name')}) "
                f"is the worst at {100*w['late_rate']:.1f}% late across {w['orders']} orders, against "
                f"a fleet average of {100*sup['fleet_late_rate']:.1f}%. Average slippage when it runs "
                f"late is {w.get('avg_delay_days_when_late')} days.")
        else:
            finding = (
                "কোনো সাপ্লায়ার আলাদাভাবে খারাপ নয়। সামগ্রিক দেরির হার {:.1f}% এবং সবচেয়ে খারাপ "
                "সাপ্লায়ার {} ({}) মাত্র {:.1f}% — পার্থক্য {} পয়েন্ট, যা {} টি অর্ডারের এই নমুনায় "
                "অর্থবহ নয়। অর্থাৎ দেরি একটি নির্দিষ্ট সাপ্লায়ারের সমস্যা নয়, পুরো প্রক্রিয়ার "
                "বৈশিষ্ট্য। সাপ্লায়ার বদলে এটি ঠিক হবে না।"
            ).format(100 * sup["fleet_late_rate"], w["supplier_id"], w.get("supplier_name"),
                     100 * w["late_rate"], sup.get("late_rate_spread_pp"),
                     sup.get("records_scanned")) if bn else (
                f"No supplier stands out as the problem. The fleet-wide late rate is "
                f"{100*sup['fleet_late_rate']:.1f}% and the worst performer, {w['supplier_id']} "
                f"({w.get('supplier_name')}), sits at just {100*w['late_rate']:.1f}% — a spread of "
                f"{sup.get('late_rate_spread_pp')} percentage points across {sup['supplier_count']} "
                f"suppliers and {sup.get('records_scanned'):,} delivered lines. That gap is not "
                f"meaningful at this sample size. Lateness here is a property of the fulfilment "
                f"process as a whole, not of any one vendor, so switching suppliers would not fix it. "
                f"When {w['supplier_id']} does slip it averages {w.get('avg_delay_days_when_late')} "
                f"days, which is the more useful number for setting delivery promises.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["suppliers", "inventory"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": (f"{s['supplier_id']} দেরির হার" if bn else f"{s['supplier_id']} late rate"),
                     "magnitude": f"{100*s['late_rate']:.1f}%"} for s in worst[:3]
                ] + [{"factor": "সামগ্রিক গড়" if bn else "Fleet average late rate",
                      "magnitude": f"{100*sup['fleet_late_rate']:.1f}%"}],
                "evidence": [_ev("suppliers", [s["supplier_id"] for s in worst]),
                             _ev("orders", sup.get("sample_late_order_ids", []))],
                "recommendedAction": {
                    "action": ((f"{w['supplier_id']} এর সাথে লিড টাইম পুনরালোচনা করুন" if bn else
                                f"Put {w['supplier_id']} on a 30-day performance review and add buffer "
                                f"stock on its highest-revenue SKUs")
                               if spread_material else
                               ("সাপ্লায়ার বদলাবেন না; ডেলিভারি প্রতিশ্রুতির সময় বাস্তবসম্মত করুন"
                                if bn else
                                f"Do not re-source on this evidence. Widen the promised delivery "
                                f"window to absorb the {100*sup['fleet_late_rate']:.1f}% baseline, and "
                                f"re-check supplier ranking after another quarter of data")),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(sup.get("records_scanned", 0),
                                             0.9 if not spread_material else w["late_rate"] * 2),
                    "requiresApproval": bool(spread_material)}}

    if family == "supplier_single":
        pool = {s["supplier_id"]: s for s in (sup.get("worst_suppliers", []) +
                                              sup.get("best_suppliers", []))}
        s = pool.get(entity)
        if not s:
            return None
        better = s["late_rate"] < sup["fleet_late_rate"]
        stated = s.get("stated_reliability")
        gap_note = ""
        if stated is not None:
            diff = stated - s["on_time_rate"]
            if abs(diff) > 0.05:
                gap_note = (f" চুক্তিতে নির্ভরযোগ্যতা {stated:.2f} বলা হলেও বাস্তবে {s['on_time_rate']:.2f} — "
                            f"{100*diff:+.1f} পয়েন্ট পার্থক্য।" if bn else
                            f" Its contract states {stated:.2f} reliability but delivered on-time rate "
                            f"is {s['on_time_rate']:.2f}, a shortfall of {100*diff:.1f} points.")
        finding = (
            "{} ({}) {} টি SKU সরবরাহ করে, {} টি অর্ডার পরিচালনা করেছে। সময়মতো ডেলিভারি {:.1f}%, "
            "সামগ্রিক গড় {:.1f}%। দেরির ক্ষেত্রে গড় বিলম্ব {} দিন এবং সর্বোচ্চ {} দিন। "
            "এই সাপ্লায়ারের মাধ্যমে {} রাজস্ব যায়।{}"
        ).format(s["supplier_id"], s.get("supplier_name"), s["skus"], s["orders"],
                 100 * s["on_time_rate"], 100 * (1 - sup["fleet_late_rate"]),
                 s.get("avg_delay_days_when_late"), s.get("worst_delay_days"),
                 bdt(s["revenue_bdt"]), gap_note) if bn else (
            f"{s['supplier_id']} ({s.get('supplier_name')}) supplies {s['skus']} SKUs across "
            f"{s['orders']} orders and delivers on time {100*s['on_time_rate']:.1f}% of the time "
            f"against a fleet average of {100*(1-sup['fleet_late_rate']):.1f}%. When it does slip, it "
            f"averages {s.get('avg_delay_days_when_late')} days late with a worst case of "
            f"{s.get('worst_delay_days')} days. {bdt(s['revenue_bdt'])} of revenue flows through it."
            + gap_note)
        return {"agent": agent, "analyzed": analyzed_block(facts, ["suppliers", "policy"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "সময়মতো ডেলিভারি" if bn else "On-time delivery rate",
                     "magnitude": f"{100*s['on_time_rate']:.1f}%"},
                    {"factor": "গড় বিলম্ব" if bn else "Average slippage when late",
                     "magnitude": f"{s.get('avg_delay_days_when_late') or 0:.1f}d"},
                    {"factor": "রাজস্ব নির্ভরতা" if bn else "Revenue dependency",
                     "magnitude": bdt(s["revenue_bdt"])}],
                "evidence": [_ev("suppliers", [s["supplier_id"]]),
                             _ev("orders", sup.get("sample_late_order_ids", []))],
                "recommendedAction": {
                    "action": (("বর্তমান শর্তে চুক্তি বহাল রাখুন" if better else
                                "লিড টাইম বাফার বাড়ান এবং চুক্তির শর্ত পুনরালোচনা করুন")
                               if bn else
                               (f"Keep {s['supplier_id']} on current terms; it performs at or above "
                                f"the fleet baseline"
                                if better else
                                f"Add buffer stock on {s['supplier_id']} SKUs and raise the "
                                f"reliability gap at the next contract review")),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(s["orders"], 0.7),
                    "requiresApproval": False}}

    # ---------------- reviews ----------------
    if family == "reviews_sentiment":
        if not rvw:
            return None
        m = rvw.get("monthly") or []
        trend = ""
        if len(m) >= 2:
            d = m[-1]["avg_score"] - m[0]["avg_score"]
            trend = (f" গত {len(m)} মাসে গড় স্কোর {m[0]['avg_score']} থেকে {m[-1]['avg_score']} — কার্যত অপরিবর্তিত।"
                     if bn else
                     f" Over the last {len(m)} months the average has gone from {m[0]['avg_score']} to "
                     f"{m[-1]['avg_score']}, which is flat.")
        finding = (
            "{} টি রিভিউয়ের গড় স্কোর {} এবং {:.1f}% গ্রাহক ২ বা তার কম দিয়েছেন, {:.1f}% দিয়েছেন ৪ বা তার বেশি।{}"
        ).format(rvw["review_count"], rvw["avg_score"], 100 * rvw["detractor_rate"],
                 100 * rvw["promoter_rate"], trend) if bn else (
            f"Across {rvw['review_count']:,} reviews the average score is {rvw['avg_score']} out of 5. "
            f"{100*rvw['detractor_rate']:.1f}% of customers leave two stars or fewer and "
            f"{100*rvw['promoter_rate']:.1f}% leave four or more.{trend} A one-in-five detractor rate "
            f"is high enough to matter, and it is not moving on its own.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["reviews", "customers"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "নেতিবাচক রিভিউয়ের হার" if bn else "Detractor share of all reviews",
                     "magnitude": f"{100*rvw['detractor_rate']:.1f}%"},
                    {"factor": "ইতিবাচক রিভিউয়ের হার" if bn else "Promoter share",
                     "magnitude": f"{100*rvw['promoter_rate']:.1f}%"}],
                "evidence": [_ev("reviews", rvw.get("sample_detractor_review_ids", []))],
                "recommendedAction": {
                    "action": ("নেতিবাচক রিভিউগুলোর টেক্সট পর্যালোচনা করুন — সংখ্যাগত তথ্যে কারণ নেই"
                               if bn else
                               "Read the comment text on the detractor reviews. The structured fields "
                               "do not explain the one-in-five detractor rate, so the reason is in the "
                               "free text, not in the delivery or product tables"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(rvw.get("records_scanned", 0), 0.7),
                    "requiresApproval": False}}

    if family == "reviews_cause":
        if not rvw:
            return None
        explains = rvw.get("delivery_explains_sentiment")
        worst = rvw.get("worst_products") or []
        if explains:
            finding = (
                "নেতিবাচক রিভিউয়ের মূল কারণ ডেলিভারি। দেরি হলে গড় স্কোর {} বনাম সময়মতো {} — "
                "পার্থক্য {} পয়েন্ট, সম্পর্ক {}।"
            ).format(rvw["avg_score_when_late"], rvw["avg_score_when_on_time"],
                     rvw["score_gap_late_vs_ontime"], rvw["late_score_correlation"]) if bn else (
                f"Delivery timing explains the negative reviews. Late orders average "
                f"{rvw['avg_score_when_late']} stars against {rvw['avg_score_when_on_time']} for "
                f"on-time ones, a gap of {rvw['score_gap_late_vs_ontime']} points with a correlation "
                f"of {rvw['late_score_correlation']}.")
        else:
            finding = (
                "দেরিতে ডেলিভারি এখানে কারণ নয়, যদিও সেটাই সবচেয়ে স্বাভাবিক অনুমান। দেরি হওয়া "
                "অর্ডারের গড় স্কোর {} এবং সময়মতো পাওয়া অর্ডারের {} — পার্থক্য মাত্র {} পয়েন্ট, "
                "আর সম্পর্ক {} অর্থাৎ কার্যত শূন্য। মাত্র {:.1f}% নেতিবাচক রিভিউ দেরি হওয়া অর্ডার থেকে এসেছে। "
                "পণ্যভিত্তিক পার্থক্যও নেই — সবচেয়ে খারাপ {} এবং সবচেয়ে ভালো পণ্যের মধ্যে ব্যবধান সামান্য। "
                "অর্থাৎ {:.1f}% অসন্তুষ্টির কারণ এই টেবিলগুলোতে নেই; রিভিউ টেক্সট পড়া দরকার।"
            ).format(rvw["avg_score_when_late"], rvw["avg_score_when_on_time"],
                     rvw["score_gap_late_vs_ontime"], rvw["late_score_correlation"],
                     100 * rvw["share_of_detractors_that_were_late"],
                     worst[0]["product_id"] if worst else "n/a",
                     100 * rvw["detractor_rate"]) if bn else (
                f"Late delivery is not the cause, even though it is the obvious hypothesis. Late "
                f"orders average {rvw['avg_score_when_late']} stars and on-time orders average "
                f"{rvw['avg_score_when_on_time']} — a gap of {rvw['score_gap_late_vs_ontime']} points "
                f"with a correlation of {rvw['late_score_correlation']}, which is indistinguishable "
                f"from zero. Only {100*rvw['share_of_detractors_that_were_late']:.1f}% of one- and "
                f"two-star reviews came from a late order. Product identity does not explain it "
                f"either: the worst-rated item "
                f"({worst[0]['product_id'] if worst else 'n/a'} at "
                f"{worst[0]['avg_score'] if worst else 'n/a'}) is barely below the best. "
                f"So the {100*rvw['detractor_rate']:.1f}% detractor rate is real but its driver is "
                f"not in the structured tables — it is in the review comment text.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["reviews", "suppliers", "revenue"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": ("দেরি ও স্কোরের সম্পর্ক" if bn else "Correlation between lateness and score"),
                     "magnitude": f"{rvw['late_score_correlation']:+.3f}"},
                    {"factor": ("দেরি হওয়া নেতিবাচক রিভিউ" if bn else "Detractors from late orders"),
                     "magnitude": f"{100*rvw['share_of_detractors_that_were_late']:.1f}%"},
                    {"factor": ("স্কোরের পার্থক্য" if bn else "Score gap late vs on-time"),
                     "magnitude": f"{rvw['score_gap_late_vs_ontime']:+.2f} pts"}],
                "evidence": [_ev("reviews", rvw.get("sample_detractor_review_ids", [])),
                             _ev("orders", sup.get("sample_late_order_ids", []))],
                "recommendedAction": {
                    "action": (("দেরি কমানোর প্রকল্পে বিনিয়োগ করবেন না; নেতিবাচক রিভিউয়ের টেক্সট "
                                "শ্রেণিবদ্ধ করে প্রকৃত কারণ বের করুন")
                               if bn else
                               "Do not fund a delivery-speed project on this evidence; it would not "
                               "move ratings. Classify the free text on the detractor reviews to find "
                               "the actual driver first") if not explains else
                              ("দেরি হওয়া অর্ডারে আগেই বার্তা পাঠান এবং ক্ষতিপূরণ দিন" if bn else
                               "Trigger a proactive delay notification and recovery offer on any order "
                               "that passes its estimated date"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(rvw.get("records_scanned", 0), 0.9),
                    "requiresApproval": False}}

    if family == "reviews_products":
        worst = rvw.get("worst_products") or []
        best = rvw.get("best_products") or []
        if not worst:
            return None
        spread = (best[-1]["avg_score"] - worst[0]["avg_score"]) if best else 0
        finding = (
            "সবচেয়ে কম রেটিং {} ({}) — {} স্কোর, {} টি রিভিউ। তবে সর্বোচ্চ ও সর্বনিম্নের ব্যবধান মাত্র "
            "{:.2f} পয়েন্ট, তাই কোনো পণ্যকে আলাদাভাবে দায়ী করা যায় না। সামগ্রিক গড় {}।"
        ).format(worst[0]["product_id"], worst[0].get("product_name"), worst[0]["avg_score"],
                 worst[0]["reviews"], spread, rvw["avg_score"]) if bn else (
            f"The lowest-rated item is {worst[0]['product_id']} ({worst[0].get('product_name')}) at "
            f"{worst[0]['avg_score']} across {worst[0]['reviews']} reviews. But the spread between "
            f"best and worst product is only {spread:.2f} points against a catalogue average of "
            f"{rvw['avg_score']}, so no individual product is dragging satisfaction down. "
            f"Dissatisfaction is spread evenly across the catalogue, which points at something "
            f"systemic rather than a quality problem with specific items.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["reviews"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": f"{w['product_id']} ({w.get('product_name')})",
                     "magnitude": f"{w['avg_score']:.2f}"} for w in worst[:3]
                ] + [{"factor": "সর্বোচ্চ-সর্বনিম্ন ব্যবধান" if bn else "Best-to-worst spread",
                      "magnitude": f"{spread:.2f} pts"}],
                "evidence": [_ev("products", [w["product_id"] for w in worst]),
                             _ev("reviews", rvw.get("sample_detractor_review_ids", []))],
                "recommendedAction": {
                    "action": ("নির্দিষ্ট পণ্য নয়, পুরো ক্যাটালগে সাধারণ কারণ খুঁজুন"
                               if bn else
                               "Do not delist or re-source individual SKUs on this data. Look for a "
                               "catalogue-wide cause such as packaging, description accuracy or "
                               "expectation setting"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(rvw.get("records_scanned", 0), 0.75),
                    "requiresApproval": False}}

    # ---------------- customers ----------------
    if family == "customer_churn":
        lap = cus.get("top_lapsed") or []
        if not lap:
            return None
        finding = (
            "গত ৯০ দিনে {} জন গ্রাহক সক্রিয় ছিলেন। তার আগের ৯০ দিনে কেনাকাটা করা {} জন আর ফেরেননি, "
            "যাদের মোট আজীবন মূল্য {}। শীর্ষে {} — {} টি অর্ডার, {} মূল্য, শেষ অর্ডার {} দিন আগে। "
            "মোট {} জন গ্রাহকের তুলনায় চার্ন {:.1f}%, যা কম।"
        ).format(cus["active_90d"], cus["lapsed_count"], bdt(cus["lapsed_revenue_bdt"]),
                 lap[0]["customer_id"], lap[0]["orders"], bdt(lap[0]["revenue_bdt"]),
                 lap[0]["recency_days"], cus["customer_count"],
                 100 * cus["lapsed_count"] / max(cus["customer_count"], 1)) if bn else (
            f"{cus['active_90d']} customers ordered in the last 90 days. {cus['lapsed_count']} who "
            f"bought in the preceding 90-day window have not come back, carrying "
            f"{bdt(cus['lapsed_revenue_bdt'])} of lifetime value between them. The most valuable is "
            f"{lap[0]['customer_id']} with {lap[0]['orders']} orders worth "
            f"{bdt(lap[0]['revenue_bdt'])}, last seen {lap[0]['recency_days']} days ago. In "
            f"proportion this is a "
            f"{100*cus['lapsed_count']/max(cus['customer_count'],1):.1f}% lapse rate against "
            f"{cus['customer_count']} customers — small, but concentrated in accounts worth "
            f"{bdt(cus['lapsed_revenue_bdt'] / max(cus['lapsed_count'],1))} each on average.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["customers", "reviews"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": (f"{c['customer_id']} — {c['recency_days']} দিন নিষ্ক্রিয়" if bn else
                                f"{c['customer_id']} inactive {c['recency_days']} days"),
                     "magnitude": bdt(c["revenue_bdt"])} for c in lap[:4]],
                "evidence": [_ev("customers", [c["customer_id"] for c in lap])],
                "recommendedAction": {
                    "action": ("শীর্ষ {} জন নিষ্ক্রিয় গ্রাহকের জন্য ব্যক্তিগত উইন-ব্যাক বার্তা পাঠান".format(min(20, cus["lapsed_count"]))
                               if bn else
                               f"Run a personal win-back on the {min(20, cus['lapsed_count'])} highest-value "
                               f"lapsed accounts. At this volume a manual message will beat a campaign"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(cus.get("records_scanned", 0), 0.7),
                    "requiresApproval": True}}

    if family == "customer_segment":
        if not cus:
            return None
        segs = cus.get("segment_value") or []
        seg_line = ""
        if segs:
            top = max(segs, key=lambda s: s.get("revenue_bdt", 0))
            seg_line = (f" সবচেয়ে বড় সেগমেন্ট {top['segment']} — {top['customers']} জন গ্রাহক, "
                        f"{bdt(top['revenue_bdt'])} রাজস্ব।" if bn else
                        f" The largest by revenue is '{top['segment']}' with {top['customers']} "
                        f"customers and {bdt(top['revenue_bdt'])}.")
        finding = (
            "মোট {} জন গ্রাহক, রিপিট ক্রয়ের হার {:.0f}% এবং গড়ে {} টি অর্ডার প্রতি গ্রাহক। "
            "গড় আজীবন মূল্য {}। সেগমেন্ট বণ্টন: {}।{} লক্ষণীয় যে প্রতিটি সেগমেন্টের গড় অর্ডার সংখ্যা "
            "প্রায় সমান, তাই সেগমেন্ট লেবেলগুলো আচরণগত পার্থক্য দেখাচ্ছে না।"
        ).format(cus["customer_count"], 100 * cus["repeat_rate"], cus["avg_orders_per_customer"],
                 bdt(cus["avg_ltv_bdt"]), json.dumps(cus.get("segment_mix", {}), ensure_ascii=False),
                 seg_line) if bn else (
            f"The base is {cus['customer_count']} customers with a {100*cus['repeat_rate']:.0f}% "
            f"repeat rate, {cus['avg_orders_per_customer']} orders each on average and "
            f"{bdt(cus['avg_ltv_bdt'])} average lifetime value. Segment mix is "
            f"{json.dumps(cus.get('segment_mix', {}))}.{seg_line} Worth flagging: average order count "
            f"is near-identical across every segment, so the segment labels are not currently "
            f"separating customers by behaviour. Any targeting built on them will behave like "
            f"targeting at random.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["customers", "revenue"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "রিপিট ক্রয়ের হার" if bn else "Repeat purchase rate",
                     "magnitude": f"{100*cus['repeat_rate']:.0f}%"},
                    {"factor": "নিষ্ক্রিয় গ্রাহকের অনুপাত" if bn else "Lapsed share of the base",
                     "magnitude": f"{100*cus['lapsed_count']/max(cus['customer_count'],1):.1f}%"},
                    {"factor": "গড় আজীবন মূল্য" if bn else "Average lifetime value",
                     "magnitude": bdt(cus["avg_ltv_bdt"])}],
                "evidence": [_ev("customers", [c["customer_id"] for c in cus.get("top_lapsed", [])])],
                "recommendedAction": {
                    "action": ("সেগমেন্ট লেবেলের বদলে সাম্প্রতিকতা ও মূল্যের ভিত্তিতে নতুন সেগমেন্ট তৈরি করুন"
                               if bn else
                               "Rebuild segments from recency and value rather than the existing "
                               "labels, which do not separate behaviour in this data"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(cus.get("records_scanned", 0), 0.7),
                    "requiresApproval": False}}

    if family == "promo_recommend":
        if not cus or not rev:
            return None
        finding = (
            "প্রমোশনের সবচেয়ে ভালো লক্ষ্য {} জন নিষ্ক্রিয় গ্রাহক, যাদের প্রমাণিত মূল্য {}। "
            "মনে রাখুন, রাজস্ব {} এবং গড় অর্ডার মূল্য {} — অর্থাৎ দাম নিয়ে সমস্যা নেই, তাই ছাড় "
            "দেওয়ার আগে ভাবুন। মার্জিন {:.1f}% হওয়ায় {}% ছাড় দিলে লাভের প্রায় {:.0f}% চলে যায়।"
        ).format(cus["lapsed_count"], bdt(cus["lapsed_revenue_bdt"]), pct(rev["delta_pct"]),
                 bdt(rev["current_aov_bdt"]), 100 * (fin.get("current_margin_pct") or 0),
                 pol.get("discount_cap_pct_without_approval", 20),
                 100 * pol.get("discount_cap_pct_without_approval", 20) /
                 max(100 * (fin.get("current_margin_pct") or 0.43), 1)) if bn else (
            f"The best promotional target is reactivation: {cus['lapsed_count']} lapsed customers "
            f"carrying {bdt(cus['lapsed_revenue_bdt'])} of proven value, which is cheaper to recover "
            f"than equivalent new demand. One caution before discounting — revenue is "
            f"{pct(rev['delta_pct'])} and average order value is holding at "
            f"{bdt(rev['current_aov_bdt'])}, so there is no evidence customers are balking at price. "
            f"At a {100*(fin.get('current_margin_pct') or 0):.1f}% gross margin, the "
            f"{pol.get('discount_cap_pct_without_approval', 20)}% discount you can approve without "
            f"sign-off would surrender roughly "
            f"{100*pol.get('discount_cap_pct_without_approval',20)/max(100*(fin.get('current_margin_pct') or 0.43),1):.0f}% "
            f"of the profit on every order it touches.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["customers", "revenue", "reviews"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "নিষ্ক্রিয় গ্রাহকের পুনরুদ্ধারযোগ্য মূল্য" if bn else "Recoverable value in lapsed accounts",
                     "magnitude": bdt(cus["lapsed_revenue_bdt"])},
                    {"factor": "গড় অর্ডার মূল্য স্থিতিশীল" if bn else "Average order value stability",
                     "magnitude": f"{rev.get('aov_change_pct', 0):+.1f}%"},
                    {"factor": "গ্রস মার্জিন" if bn else "Gross margin headroom",
                     "magnitude": f"{100*(fin.get('current_margin_pct') or 0):.1f}%"}],
                "evidence": [_ev("customers", [c["customer_id"] for c in cus.get("top_lapsed", [])]),
                             _ev("orders", rev.get("sample_order_ids_current", []))],
                "recommendedAction": {
                    "action": ("ছাড়ের বদলে নিষ্ক্রিয় গ্রাহকদের ব্যক্তিগত বার্তা পাঠান এবং একটি কন্ট্রোল গ্রুপ রাখুন"
                               if bn else
                               "Lead with a personal reactivation message rather than a discount, and "
                               "hold a control group so you can measure whether the offer added anything"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(cus.get("records_scanned", 0), 0.65),
                    "requiresApproval": True}}

    # ---------------- finance ----------------
    if family == "finance_margin":
        if fin.get("current_margin_pct") is None:
            return None
        chg = fin.get("margin_change_pp") or 0
        stable = abs(chg) < 1.0
        finding = (
            "{} মাসে রাজস্ব {}, পণ্যের ক্রয়মূল্য {}, গ্রস মার্জিন {} অর্থাৎ {:.1f}%। আগের মাসে ছিল "
            "{:.1f}%, পরিবর্তন {} পয়েন্ট — {}। ফ্রেইট খরচ রাজস্বের {:.2f}%, যা খুবই কম। "
            "মুলতুবি রিস্টক প্রতিশ্রুতি {} নগদ প্রবাহের প্রধান চাপ।"
        ).format(fin["current_month"], bdt(fin["current_revenue_bdt"]), bdt(fin["current_cogs_bdt"]),
                 bdt(fin["current_gross_margin_bdt"]), 100 * fin["current_margin_pct"],
                 100 * (fin.get("previous_margin_pct") or 0), f"{chg:+.2f}",
                 "কার্যত অপরিবর্তিত" if stable else "উল্লেখযোগ্য",
                 100 * (fin.get("current_freight_ratio") or 0),
                 bdt(fin.get("open_reorder_commitment_bdt"))) if bn else (
            f"In {fin['current_month']}, revenue of {bdt(fin['current_revenue_bdt'])} against COGS of "
            f"{bdt(fin['current_cogs_bdt'])} produced {bdt(fin['current_gross_margin_bdt'])} of gross "
            f"margin, or {100*fin['current_margin_pct']:.1f}%. That compares with "
            f"{100*(fin.get('previous_margin_pct') or 0):.1f}% the month before — a move of "
            f"{chg:+.2f} points, which is "
            f"{'noise, not a trend' if stable else 'worth watching'}. Freight is running at "
            f"{100*(fin.get('current_freight_ratio') or 0):.2f}% of revenue, low enough to ignore. "
            f"The real pressure on cash is the {bdt(fin.get('open_reorder_commitment_bdt'))} of open "
            f"restock commitment, which is several times a month of gross margin.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["finance", "revenue", "inventory"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "মার্জিন পরিবর্তন" if bn else "Month-over-month margin movement",
                     "magnitude": f"{chg:+.2f}pp"},
                    {"factor": "ফ্রেইট অনুপাত" if bn else "Freight as share of revenue",
                     "magnitude": f"{100*(fin.get('current_freight_ratio') or 0):.2f}%"},
                    {"factor": "মুলতুবি রিস্টক প্রতিশ্রুতি" if bn else "Open restock commitment against cash",
                     "magnitude": bdt(fin.get("open_reorder_commitment_bdt"))}],
                "evidence": [_ev("payments", rev.get("sample_order_ids_current", [])),
                             _ev("inventory", [c["product_id"] for c in inv.get("critical", [])])],
                "recommendedAction": {
                    "action": ("মার্জিন স্থিতিশীল; রিস্টক অর্ডার ধাপে ধাপে দিয়ে নগদ প্রবাহ রক্ষা করুন"
                               if bn else
                               "Margin needs no intervention. Stage the restock commitment across "
                               "cycles so it does not consume more than one month of gross margin at once"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(fin.get("records_scanned", 0), 0.75),
                    "requiresApproval": True}}

    if family == "finance_thin":
        weakest = fin.get("weakest_margin_skus") or []
        if not weakest:
            return None
        thin_n = fin.get("thin_margin_count", 0)
        w = weakest[0]
        if thin_n:
            finding = (
                "{} টি SKU এর মার্জিন {}% এর নিচে। সবচেয়ে খারাপ {} — {:.1f}% মার্জিন, {} রাজস্ব।"
            ).format(thin_n, int(100 * MATERIALITY["thin_margin_pct"]), w["product_id"],
                     100 * w["margin_pct"], bdt(w["revenue_bdt"])) if bn else (
                f"{thin_n} SKUs are running below a {100*MATERIALITY['thin_margin_pct']:.0f}% gross "
                f"margin. The weakest is {w['product_id']} at {100*w['margin_pct']:.1f}% on "
                f"{bdt(w['revenue_bdt'])} of revenue.")
        else:
            finding = (
                "কোনো পণ্যের মার্জিন উদ্বেগজনক নয়। পোর্টফোলিও গড় {:.1f}% এবং সবচেয়ে দুর্বল SKU "
                "{} ও {:.1f}% মার্জিন ধরে রেখেছে — {}% সীমার অনেক উপরে। {} টি SKU এর কোনোটিই "
                "লোকসানে বিক্রি হচ্ছে না, তাই দাম পুনর্নির্ধারণের প্রয়োজন নেই। "
                "মূল্যনির্ধারণ এখানে সমস্যা নয়, মনোযোগ স্টক প্রাপ্যতায় দেওয়া উচিত।"
            ).format(100 * (fin.get("portfolio_margin_pct") or 0), w["product_id"],
                     100 * w["margin_pct"], int(100 * MATERIALITY["thin_margin_pct"]),
                     inv.get("total_skus")) if bn else (
                f"No product has a margin problem. The portfolio runs at "
                f"{100*(fin.get('portfolio_margin_pct') or 0):.1f}% gross margin and even the weakest "
                f"SKU, {w['product_id']}, holds {100*w['margin_pct']:.1f}% on {int(w['units'])} units "
                f"— comfortably above the {100*MATERIALITY['thin_margin_pct']:.0f}% line where "
                f"re-pricing would be worth the disruption. Nothing in the catalogue is being sold at "
                f"a loss. Pricing is not where the money is leaking here; stock availability is.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["finance", "inventory"]),
                "finding": finding,
                "contributingFactors": [{"factor": s["product_id"], "magnitude": f"{100*s['margin_pct']:.1f}%"}
                                        for s in weakest[:3]
                                        ] + [{"factor": "পোর্টফোলিও মার্জিন" if bn else "Portfolio gross margin",
                                              "magnitude": f"{100*(fin.get('portfolio_margin_pct') or 0):.1f}%"}],
                "evidence": [_ev("order_items", [s["product_id"] for s in weakest]),
                             _ev("inventory", [s["product_id"] for s in weakest])],
                "recommendedAction": {
                    "action": (("মূল্য পুনর্নির্ধারণের দরকার নেই; স্টক প্রাপ্যতায় মনোযোগ দিন" if bn else
                                "No re-pricing needed. Redirect the attention to stock availability, "
                                "which is costing more than pricing is") if not thin_n else
                               (f"{w['product_id']} এর দাম বা সাপ্লায়ার খরচ পুনরালোচনা করুন" if bn else
                                f"Re-price or renegotiate supply cost on {w['product_id']}")),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(fin.get("records_scanned", 0), 0.85),
                    "requiresApproval": bool(thin_n)}}

    if family == "finance_payments":
        mix = fin.get("payment_mix") or {}
        if not mix:
            return None
        top = sorted(mix.items(), key=lambda x: -x[1])
        digital = sum(v for k, v in mix.items() if k in ("bkash", "nagad"))
        cod = mix.get("cash_on_delivery", 0)
        finding = (
            "পেমেন্টের {:.0f}% মোবাইল ওয়ালেটে (bKash {:.0f}%, Nagad {:.0f}%) এবং {:.0f}% ক্যাশ অন "
            "ডেলিভারি। ফ্রেইট খরচ রাজস্বের {:.2f}%। ক্যাশ অন ডেলিভারি অংশ বড় হওয়ায় বাতিল ও "
            "ফেরতের ঝুঁকি নগদ প্রবাহে সরাসরি প্রভাব ফেলে।"
        ).format(100 * digital, 100 * mix.get("bkash", 0), 100 * mix.get("nagad", 0), 100 * cod,
                 100 * (fin.get("current_freight_ratio") or 0)) if bn else (
            f"{100*digital:.0f}% of payment value comes through mobile wallets "
            f"(bKash {100*mix.get('bkash',0):.0f}%, Nagad {100*mix.get('nagad',0):.0f}%), with "
            f"{100*cod:.0f}% on cash on delivery and the remainder split between cards and bank "
            f"transfer. Freight runs at {100*(fin.get('current_freight_ratio') or 0):.2f}% of revenue. "
            f"The cash-on-delivery share is the number to watch: at {100*cod:.0f}% of value it ties "
            f"working capital to delivery success, so every failed or refused delivery hits cash "
            f"directly rather than just margin.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["finance", "revenue"]),
                "finding": finding,
                "contributingFactors": [{"factor": k, "magnitude": f"{100*v:.1f}%"} for k, v in top[:4]],
                "evidence": [_ev("payments", rev.get("sample_order_ids_current", []))],
                "recommendedAction": {
                    "action": ("ক্যাশ অন ডেলিভারি অর্ডারে প্রি-পেমেন্টে ছোট প্রণোদনা দিন"
                               if bn else
                               "Offer a small incentive for prepayment on cash-on-delivery orders to "
                               "shorten the cash cycle; keep it well inside the "
                               f"{pol.get('discount_cap_pct_without_approval', 20)}% discount cap"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(fin.get("records_scanned", 0), 0.7),
                    "requiresApproval": False}}

    # ---------------- policy & docs ----------------
    if family == "policy_approval":
        if not pol:
            return None
        pending = pol.get("pending_reorder_value_bdt") or 0
        thr = pol.get("high_value_threshold_bdt") or 5000
        exceeds = pol.get("pending_reorder_exceeds_threshold")
        finding = (
            "নীতি অনুযায়ী {} এর বেশি যেকোনো আর্থিক সিদ্ধান্তে অনুমোদন লাগে, এবং পারচেজ অর্ডারে "
            "সবসময়ই অনুমোদন প্রয়োজন। বর্তমানে মুলতুবি রিস্টকের মূল্য {}, যা সীমার {} গুণ। "
            "সুতরাং এটি স্বয়ংক্রিয়ভাবে চালানো যাবে না — মালিকের অনুমোদন বাধ্যতামূলক।"
        ).format(bdt(thr), bdt(pending), pol.get("threshold_multiple")) if bn else (
            f"Policy sets a {bdt(thr)} ceiling on financial actions without sign-off, and purchase "
            f"orders require approval unconditionally regardless of value. The pending restock is "
            f"{bdt(pending)}, roughly {pol.get('threshold_multiple')}x the ceiling. Both rules bite, "
            f"so this cannot execute automatically — owner approval is mandatory before any PO is "
            f"sent. Customer messages can be drafted automatically but not sent, and destructive "
            f"actions always need sign-off.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["policy", "inventory", "finance"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "উচ্চমূল্য সীমা" if bn else "High-value action threshold",
                     "magnitude": bdt(thr)},
                    {"factor": "মুলতুবি অর্ডারের মূল্য" if bn else "Pending reorder value",
                     "magnitude": bdt(pending)},
                    {"factor": "পারচেজ অর্ডার নীতি" if bn else "Purchase order rule",
                     "magnitude": "always requires approval"}],
                "evidence": [_ev("inventory", [c["product_id"] for c in inv.get("critical", [])]),
                             _ev("suppliers", [s["supplier_id"] for s in pol.get("supplier_contracts", [])])],
                "recommendedAction": {
                    "action": ("অনুমোদনের জন্য মালিকের কাছে পাঠান; সীমা অতিক্রম করায় স্বয়ংক্রিয় অনুমোদন সম্ভব নয়"
                               if bn else
                               f"Route to the owner for sign-off. If you want part of this to flow "
                               f"automatically, split the batch so each PO sits under {bdt(thr)} — "
                               f"though the purchase-order rule would still require approval"),
                    "riskTier": "high" if exceeds else "low",
                    "confidence": 0.95,
                    "requiresApproval": True}}

    if family == "policy_sla":
        if not pol:
            return None
        breaches = pol.get("sla_breaches") or []
        contracts = pol.get("supplier_contracts") or []
        if breaches:
            b = breaches[0]
            finding = (
                "{} টি সাপ্লায়ার চুক্তিতে বলা নির্ভরযোগ্যতার নিচে পারফর্ম করছে। {} ({}) চুক্তিতে "
                "{:.2f} নির্ভরযোগ্যতার প্রতিশ্রুতি দিয়েছে কিন্তু বাস্তবে সময়মতো ডেলিভারি {:.2f} — "
                "ঘাটতি {} পয়েন্ট। বাকি {} টি সাপ্লায়ার চুক্তির সীমার মধ্যে আছে।"
            ).format(len(breaches), b["supplier_id"], b.get("supplier_name"),
                     b["stated_reliability"], b["actual_on_time_rate"], b["shortfall_pp"],
                     len(contracts) - len(breaches)) if bn else (
                f"{len(breaches)} supplier is performing below its contracted reliability. "
                f"{b['supplier_id']} ({b.get('supplier_name')}) commits to {b['stated_reliability']:.2f} "
                f"in its agreement but is delivering on time at {b['actual_on_time_rate']:.2f}, a "
                f"shortfall of {b['shortfall_pp']} percentage points. The remaining "
                f"{len(contracts) - len(breaches)} suppliers are operating inside their stated terms. "
                f"This is a documented, checkable breach — the kind worth raising at a contract review "
                f"rather than absorbing quietly as buffer stock.")
        else:
            finding = (
                "কোনো সাপ্লায়ার চুক্তির শর্ত ভঙ্গ করছে না। {} টি চুক্তির প্রতিটিতে বলা নির্ভরযোগ্যতা "
                "বাস্তব পারফরম্যান্সের সাথে ৫ পয়েন্টের মধ্যে মিলছে।"
            ).format(len(contracts)) if bn else (
                f"No supplier is breaching its contract. Across {len(contracts)} agreements, stated "
                f"reliability matches delivered on-time performance within five percentage points in "
                f"every case.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["policy", "suppliers"]),
                "finding": finding,
                "contributingFactors": ([
                    {"factor": (f"{b['supplier_id']} ঘাটতি" if bn else
                                f"{b['supplier_id']} stated {b['stated_reliability']:.2f} vs actual {b['actual_on_time_rate']:.2f}"),
                     "magnitude": f"-{b['shortfall_pp']:.1f}pp"} for b in breaches[:3]]
                    or [{"factor": "চুক্তি সম্মতি" if bn else "Contract compliance across all suppliers",
                         "magnitude": "0 breaches"}]),
                "evidence": [_ev("suppliers", [b["supplier_id"] for b in breaches] or
                                 [c["supplier_id"] for c in contracts]),
                             _ev("orders", sup.get("sample_late_order_ids", []))],
                "recommendedAction": {
                    "action": ((f"{breaches[0]['supplier_id']} এর সাথে চুক্তি পর্যালোচনার বৈঠক ডাকুন" if bn else
                                f"Raise the {breaches[0]['shortfall_pp']}pp reliability shortfall with "
                                f"{breaches[0]['supplier_id']} at the next contract review and ask for "
                                f"either a revised lead time or a service credit")
                               if breaches else
                               ("বর্তমান চুক্তিগুলো বহাল রাখুন" if bn else
                                "No contract action needed; re-run this check next quarter")),
                    "riskTier": RISK_TIER[agent],
                    "confidence": confidence(sup.get("records_scanned", 0), 0.8),
                    "requiresApproval": False}}

    if family == "policy_rules":
        if not pol:
            return None
        finding = (
            "{} এর পরিচালনা নীতি: অনুমোদন ছাড়া সর্বোচ্চ {}% ছাড় দেওয়া যায় এবং {} পর্যন্ত রিফান্ড "
            "করা যায়। {} এর বেশি যেকোনো আর্থিক পদক্ষেপে অনুমোদন লাগে। পারচেজ অর্ডার ও রিস্টক "
            "সুপারিশ সবসময় অনুমোদনসাপেক্ষ। গ্রাহক বার্তা স্বয়ংক্রিয়ভাবে খসড়া করা যায় কিন্তু "
            "পাঠাতে অনুমোদন লাগে। অটোমেশন {} এবং ধ্বংসাত্মক পদক্ষেপে সর্বদা অনুমোদন প্রয়োজন।"
        ).format(pol.get("business_name"), pol.get("discount_cap_pct_without_approval"),
                 bdt(pol.get("refund_cap_bdt_without_approval")), bdt(pol.get("high_value_threshold_bdt")),
                 "চালু" if pol.get("automation_enabled") else "বন্ধ") if bn else (
            f"{pol.get('business_name')} operating policy: discounts up to "
            f"{pol.get('discount_cap_pct_without_approval')}% and refunds up to "
            f"{bdt(pol.get('refund_cap_bdt_without_approval'))} can go out without sign-off. Any "
            f"financial action above {bdt(pol.get('high_value_threshold_bdt'))} needs approval. "
            f"Purchase orders and restock recommendations always require approval regardless of "
            f"value. Customer messages may be drafted automatically but sending requires sign-off. "
            f"Automation is "
            f"{'enabled' if pol.get('automation_enabled') else 'disabled'}, with destructive actions "
            f"always gated.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["policy", "finance"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "ছাড়ের সীমা" if bn else "Discount without approval",
                     "magnitude": f"{pol.get('discount_cap_pct_without_approval')}%"},
                    {"factor": "রিফান্ড সীমা" if bn else "Refund without approval",
                     "magnitude": bdt(pol.get("refund_cap_bdt_without_approval"))},
                    {"factor": "উচ্চমূল্য সীমা" if bn else "High-value action threshold",
                     "magnitude": bdt(pol.get("high_value_threshold_bdt"))}],
                "evidence": [_ev("suppliers", [c["supplier_id"] for c in pol.get("supplier_contracts", [])])],
                "recommendedAction": {
                    "action": ("বর্তমান নীতি বহাল; রিফান্ড সীমা গড় অর্ডার মূল্যের তুলনায় কম কিনা পর্যালোচনা করুন"
                               if bn else
                               f"Worth reviewing one number: the {bdt(pol.get('refund_cap_bdt_without_approval'))} "
                               f"refund ceiling is far below the {bdt(rev.get('current_aov_bdt'))} average "
                               f"order value, so nearly every refund escalates to you"),
                    "riskTier": RISK_TIER[agent],
                    "confidence": 0.95,
                    "requiresApproval": False}}

    # ---------------- automation ----------------
    if family == "automation_po":
        crit = inv.get("critical") or []
        if not crit:
            return None
        batch = crit[:5]
        total = sum(c["reorder_cost_bdt"] for c in batch)
        thr = pol.get("high_value_threshold_bdt") or 5000
        suppliers_n = len({c["supplier_id"] for c in batch if c.get("supplier_id")})
        finding = (
            "রিঅর্ডার ওয়ার্কফ্লো {} টি ঝুঁকিপূর্ণ SKU চিহ্নিত করেছে। শীর্ষ {} টির জন্য খসড়া পারচেজ "
            "অর্ডার প্রস্তুত, মোট {}, {} টি সাপ্লায়ারে বিভক্ত। পরিমাণ লিড টাইম প্লাস ৭ দিনের বাফার "
            "অনুযায়ী হিসাব করা। নীতি অনুযায়ী পারচেজ অর্ডারে সবসময় অনুমোদন লাগে এবং এই ব্যাচ "
            "{} সীমাও অতিক্রম করেছে, তাই কিছুই পাঠানো হয়নি — সব অনুমোদনের অপেক্ষায়।"
        ).format(inv["at_risk_count"], len(batch), bdt(total), suppliers_n, bdt(thr)) if bn else (
            f"The reorder workflow identified {inv['at_risk_count']} SKUs that will stock out before "
            f"a replacement can land. Draft purchase orders are staged for the top {len(batch)} by "
            f"urgency, totalling {bdt(total)} across {suppliers_n} suppliers, with quantities sized to "
            f"lead time plus a seven day buffer. Nothing has been sent. Policy requires approval on "
            f"every purchase order regardless of value, and this batch also clears the {bdt(thr)} "
            f"high-value threshold, so all {len(batch)} orders are queued for sign-off.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["inventory", "policy", "suppliers"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": (f"{c['product_id']} — {c['reorder_qty']} ইউনিট" if bn else
                                f"{c['product_id']}: {c['reorder_qty']} units to {c.get('supplier_id')}"),
                     "magnitude": bdt(c["reorder_cost_bdt"])} for c in batch[:4]],
                "evidence": [_ev("inventory", [c["product_id"] for c in batch]),
                             _ev("suppliers", sorted({c["supplier_id"] for c in batch if c.get("supplier_id")}))],
                "recommendedAction": {
                    "action": ("{} মূল্যের {} টি খসড়া পারচেজ অর্ডার অনুমোদনের অপেক্ষায়; অনুমোদনের পর {} আগে পাঠান".format(
                        bdt(total), len(batch), batch[0]["product_id"]) if bn else
                        f"Release the {len(batch)} staged purchase orders worth {bdt(total)} once "
                        f"approved, sending {batch[0]['product_id']} first as it has the widest cover gap"),
                    "riskTier": "high" if total > thr else "medium",
                    "confidence": confidence(inv.get("records_scanned", 0) * 4, 0.9),
                    "requiresApproval": True}}

    if family == "automation_policy":
        crit = inv.get("critical") or []
        if not crit or not pol:
            return None
        total = sum(c["reorder_cost_bdt"] for c in crit[:5])
        thr = pol.get("high_value_threshold_bdt") or 5000
        auto_ok = (total <= thr) and not pol.get("purchase_order_requires_approval", True)
        finding = (
            "নীতি যাচাই সম্পন্ন। প্রস্তাবিত রিস্টকের মূল্য {}, উচ্চমূল্য সীমা {} — সীমার {}। "
            "এছাড়া পারচেজ অর্ডার নীতিতে মূল্য নির্বিশেষে অনুমোদন বাধ্যতামূলক। দুটি নিয়মের কারণেই "
            "স্বয়ংক্রিয় কার্যকর করা সম্ভব নয়।"
        ).format(bdt(total), bdt(thr), "উপরে" if total > thr else "নিচে") if bn else (
            f"Policy check complete. The proposed restock totals {bdt(total)} against a "
            f"{bdt(thr)} high-value threshold, so it sits "
            f"{'above' if total > thr else 'below'} the ceiling. Separately, the inventory policy "
            f"requires approval on every purchase order regardless of value. Either rule alone would "
            f"block automatic execution, so the workflow stops here and waits for sign-off. "
            f"Drafting the POs is permitted and has been done; sending them is not.")
        return {"agent": agent, "analyzed": analyzed_block(facts, ["policy", "inventory", "finance"]),
                "finding": finding,
                "contributingFactors": [
                    {"factor": "ব্যাচের মূল্য বনাম সীমা" if bn else "Batch value against threshold",
                     "magnitude": bdt(total)},
                    {"factor": "পারচেজ অর্ডার নীতি" if bn else "Purchase order approval rule",
                     "magnitude": "unconditional"},
                    {"factor": "প্রভাবিত SKU" if bn else "SKUs in scope",
                     "magnitude": str(len(crit[:5]))}],
                "evidence": [_ev("inventory", [c["product_id"] for c in crit[:5]])],
                "recommendedAction": {
                    "action": (("মালিকের অনুমোদনের জন্য পাঠান" if bn else
                                "Route to the owner for sign-off; drafts are ready to send on approval")
                               if not auto_ok else
                               ("স্বয়ংক্রিয়ভাবে কার্যকর করুন" if bn else
                                "Auto-approve and execute the workflow")),
                    "riskTier": "high" if total > thr else "medium",
                    "confidence": 0.95,
                    "requiresApproval": not auto_ok}}

    return None


# --------------------------------------------------------------------------
# Surface-form augmentation. These wrappers do not change meaning, so the label
# stays valid, but they teach the model that one intent arrives in many shapes.
# --------------------------------------------------------------------------
PREFIXES = {
    "en": ["", "", "Quick question - ", "Hey, ", "I need to know: ", "Can you check: ",
           "As the owner I want to understand: ", "For my weekly review: ", "Urgent: "],
    "bn": ["", "", "একটা প্রশ্ন — ", "দয়া করে বলুন, ", "আমার জানা দরকার: ",
           "সাপ্তাহিক রিভিউয়ের জন্য: ", "জরুরি: ", "একটু দেখুন তো, "],
}
SUFFIXES = {
    "en": ["", "", " Give me specifics.", " Keep it short.", " I need the evidence.",
           " Include record IDs.", " Numbers please, not opinions.", " What should I do about it?"],
    "bn": ["", "", " বিস্তারিত দিন।", " সংক্ষেপে বলুন।", " প্রমাণসহ দিন।",
           " রেকর্ড আইডি সহ দিন।", " এখন কী করা উচিত?", " সংখ্যা দিয়ে বলুন।"],
}


def _surface(q: str, lang: str, rng: random.Random) -> str:
    pre, suf = rng.choice(PREFIXES[lang]), rng.choice(SUFFIXES[lang])
    out = f"{pre}{q}{suf}".strip()
    if pre and lang == "en" and pre.endswith((": ", "- ", ", ")) and len(out) > len(pre):
        ch = out[len(pre)]
        if ch.isupper():
            out = pre + ch.lower() + out[len(pre) + 1:]
    return out


def build_dataset(fs: FactStore, target: int = 1200, seed: int = 11) -> List[Dict[str, Any]]:
    """Round-robin over families so no single agent dominates the mix."""
    rng = random.Random(seed)
    facts = fs.facts
    rows: List[Dict[str, Any]] = []
    seen = set()

    inv_facts = facts.get("inventory", {})
    sku_pool = ([c["product_id"] for c in inv_facts.get("critical", [])]
                or inv_facts.get("healthy_sample", []))
    sup_pool = [s["supplier_id"] for s in (facts.get("suppliers", {}).get("worst_suppliers", [])
                                           + facts.get("suppliers", {}).get("best_suppliers", []))]

    # Keep only families whose builder actually produces output on this dataset.
    families = []
    for fam in QUESTIONS:
        probe_entity = None
        joined = " ".join(QUESTIONS[fam]["en"])
        if "{pid}" in joined:
            probe_entity = sku_pool[0] if sku_pool else None
        elif "{sid}" in joined:
            probe_entity = sup_pool[0] if sup_pool else None
        if build_label(fam, facts, "en", probe_entity) is not None:
            families.append(fam)
    if not families:
        raise RuntimeError("No label family could be built - were facts computed?")

    per_family = target // len(families) + 1
    for fam in families:
        made, guard = 0, 0
        while made < per_family and guard < per_family * 30:
            guard += 1
            lang = "en" if (made % 2 == 0) else "bn"
            base = rng.choice(QUESTIONS[fam][lang])

            entity = None
            if "{pid}" in base:
                if not sku_pool:
                    break
                entity = rng.choice(sku_pool)
                base = base.format(pid=entity)
            if "{sid}" in base:
                if not sup_pool:
                    break
                entity = rng.choice(sup_pool)
                base = base.format(sid=entity)

            label = build_label(fam, facts, lang, entity)
            if label is None:
                break

            agent = label["agent"]
            q = _surface(base, lang, rng)
            # A quarter of rows address the agent explicitly, and half of those
            # use kebab-case, mirroring how the frontend calls
            # POST /api/agents/{agent_id}/run.
            if rng.random() < 0.25:
                tag = to_kebab(agent) if rng.random() < 0.5 else agent
                q = f"[agent: {tag}] {q}"

            key = (q, agent)
            if key in seen:
                continue
            seen.add(key)

            domains = route_domains(base, agent=agent)
            rows.append({
                "family": fam, "lang": lang, "agent": agent,
                "instruction": q,
                "input": f"FACTS:\n{render_context(facts, domains)}",
                "output": json.dumps(label, ensure_ascii=False, separators=(",", ": ")),
            })
            made += 1

    rng.shuffle(rows)
    return rows[:target] if len(rows) > target else rows


def to_alpaca(rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    return [{"instruction": r["instruction"], "input": r["input"], "output": r["output"]}
            for r in rows]


def to_sharegpt(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [{"conversations": [
        {"from": "system", "value": SYSTEM_PROMPT},
        {"from": "human", "value": f"{r['instruction']}\n\n{r['input']}"},
        {"from": "gpt", "value": r["output"]},
    ]} for r in rows]

In [ ]:
import thalamus_dataset as td
importlib.reload(td)

# Match the FACTS budget to the runtime's sequence length.
_orig = tc.render_context
tc.render_context = lambda facts, domains, budget_chars=CFG["context_budget"]: _orig(
    facts, domains, budget_chars)
td.render_context = tc.render_context

rows = td.build_dataset(fs, target=CFG["dataset_size"], seed=CFG["seed"])

import collections
print(f"rows                {len(rows):,}")
print(f"unique instructions {len(set(r['instruction'] for r in rows)):,}")
print(f"language split      {dict(collections.Counter(r['lang'] for r in rows))}")
print(f"families            {len(set(r['family'] for r in rows))} of {len(td.QUESTIONS)}")
print("agent split:")
for a, n in sorted(collections.Counter(r["agent"] for r in rows).items()):
    print(f"  {tc.to_kebab(a):<20} {n:>5}")

for r in rows:
    json.loads(r["output"])      # raises if any label is malformed
print("\nall labels parse    PASS")

In [ ]:
# The three findings that show the materiality gates working. Read these --
# they are the difference between a demo that survives a judge's follow-up
# question and one that does not.
for fam in ["revenue_why", "supplier_worst", "reviews_cause", "finance_thin", "policy_sla"]:
    ex = next((r for r in rows if r["family"] == fam and r["lang"] == "en"), None)
    if not ex:
        continue
    print(f"--- {fam} ---")
    print(json.loads(ex["output"])["finding"])
    print()

In [ ]:
bn = next(r for r in rows if r["lang"] == "bn" and r["family"] == "inventory_at_risk")
print("INSTRUCTION\n", bn["instruction"], "\n")
print("OUTPUT\n", json.dumps(json.loads(bn["output"]), ensure_ascii=False, indent=2)[:1500])

### 3.1 Dataset Export & Stratified Holdout Split

Export formatted pairs to Alpaca and ShareGPT schemas. Apply stratified splitting across query families and language types to guarantee balanced evaluation across all seven agent domains.

In [ ]:
from sklearn.model_selection import train_test_split

alpaca, sharegpt = td.to_alpaca(rows), td.to_sharegpt(rows)
json.dump(alpaca,   open("/content/thalamus_alpaca.json", "w"),   ensure_ascii=False, indent=2)
json.dump(sharegpt, open("/content/thalamus_sharegpt.json", "w"), ensure_ascii=False, indent=2)

strat = [f"{r['family']}|{r['lang']}" for r in rows]
counts = collections.Counter(strat)
strat = [s if counts[s] >= 2 else "rare" for s in strat]     # split needs >=2 per stratum

idx_tr, idx_va = train_test_split(range(len(rows)), test_size=0.06,
                                  random_state=CFG["seed"], stratify=strat)
train_rows = [sharegpt[i] for i in idx_tr]
val_rows   = [sharegpt[i] for i in idx_va]
val_raw    = [rows[i] for i in idx_va]
json.dump(val_raw, open("/content/thalamus_val_raw.json", "w"), ensure_ascii=False, indent=2)
print(f"train {len(train_rows):,} | validation {len(val_rows):,}")

## 4. QLoRA Parameter-Efficient Fine-Tuning

Configure 4-bit base model loading (Qwen-2.5-7B-Instruct or LLaMA-3.1-8B-Instruct) with Low-Rank Adaptation (LoRA) on attention and MLP projections. Supervised loss is computed strictly on assistant response tokens, preserving model capacity for structured output synthesis.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset

# Reclaim anything left from data prep before the 4-bit weights land.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["model_name"],
    max_seq_length = CFG["max_seq_length"],
    dtype          = None,          # resolved from CFG below
    load_in_4bit   = True,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = FastLanguageModel.get_peft_model(
    model,
    r                          = CFG["lora_r"],
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = CFG["lora_alpha"],
    lora_dropout               = CFG["lora_dropout"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",   # ~30% less VRAM; required on T4
    random_state               = CFG["seed"],
    use_rslora                 = False,
)

tokenizer = get_chat_template(tokenizer, chat_template=CFG["chat_template"])
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")
if torch.cuda.is_available():
    print(f"VRAM after load : {torch.cuda.memory_reserved()/1e9:.2f} GB")

In [ ]:
ROLE = {"system": "system", "human": "user", "gpt": "assistant"}

def to_text(batch):
    out = []
    for convo in batch["conversations"]:
        msgs = [{"role": ROLE[m["from"]], "content": m["value"]} for m in convo]
        out.append(tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=False))
    return {"text": out}

# num_proc omitted deliberately: forking after CUDA init raises
# "Cannot re-initialize CUDA in forked subprocess" on Colab.
train_ds = Dataset.from_list(train_rows).map(to_text, batched=True,
                                             remove_columns=["conversations"])
val_ds   = Dataset.from_list(val_rows).map(to_text, batched=True,
                                           remove_columns=["conversations"])

# Bangla tokenises 2-3x worse than English, so a budget that looks safe in
# characters can overflow in tokens. Check before spending 40 minutes on it.
lens = [len(tokenizer(t).input_ids) for t in train_ds["text"]]
over = sum(1 for L in lens if L > CFG["max_seq_length"])
print(f"token length  p50 {int(np.percentile(lens,50))} | p95 {int(np.percentile(lens,95))} "
      f"| max {max(lens)} | limit {CFG['max_seq_length']}")
print(f"truncated     {over} ({100*over/len(lens):.1f}%)")
if over:
    print("  -> lower CFG['context_budget'], rebuild the dataset, and re-run this cell")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

args_kwargs = dict(
    per_device_train_batch_size = CFG["batch_size"],
    gradient_accumulation_steps = CFG["grad_accum"],
    num_train_epochs            = CFG["epochs"],
    learning_rate               = CFG["learning_rate"],
    lr_scheduler_type           = "cosine",
    warmup_ratio                = CFG["warmup_ratio"],
    weight_decay                = CFG["weight_decay"],
    optim                       = "adamw_8bit",
    fp16                        = CFG["use_fp16"],     # explicit, not autodetected
    bf16                        = CFG["use_bf16"],
    logging_steps               = 10,
    save_strategy               = "no",
    seed                        = CFG["seed"],
    output_dir                  = CFG["output_dir"],
    report_to                   = "none",
)

# dataset_num_proc=1 everywhere: any value above 1 forks a subprocess after CUDA
# has been initialised, which is an immediate RuntimeError on Colab.
try:
    from trl import SFTConfig
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        args=SFTConfig(dataset_text_field="text", dataset_num_proc=1,
                       max_seq_length=CFG["max_seq_length"], **args_kwargs))
except (ImportError, TypeError):
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        dataset_text_field="text", max_seq_length=CFG["max_seq_length"],
        dataset_num_proc=1, packing=False,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        args=TrainingArguments(**args_kwargs))

PARTS = {
    "llama-3.1": ("<|start_header_id|>user<|end_header_id|>\n\n",
                  "<|start_header_id|>assistant<|end_header_id|>\n\n"),
    "qwen-2.5":  ("<|im_start|>user\n", "<|im_start|>assistant\n"),
}
instr_part, resp_part = PARTS[CFG["chat_template"]]
trainer = train_on_responses_only(trainer, instruction_part=instr_part, response_part=resp_part)

visible = tokenizer.decode([t for t in trainer.train_dataset[0]["labels"] if t != -100])
print("SUPERVISED SPAN (must start with '{'):\n", visible[:240])
assert visible.lstrip().startswith("{"), "response masking is wrong; check PARTS for this template"

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

stats = trainer.train()

print(f"\nfinal train loss {stats.training_loss:.4f}")
print(f"runtime          {stats.metrics['train_runtime']/60:.1f} min")
if torch.cuda.is_available():
    print(f"peak VRAM        {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

In [ ]:
model.save_pretrained(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print("adapter saved to", CFG["output_dir"])

# Optional: push to the Hub so a persistent host can pull it without Drive.
# from google.colab import userdata
# model.push_to_hub("your-org/thalamus-lora-bd", token=userdata.get("HF_TOKEN"))

## 5. Runtime Inference & Hallucination Guardrails (`thalamus_runtime.py`)

Production inference utilizes a multi-stage validation pipeline to guarantee UI-ready responses:
1. **Balanced Extraction**: Safely extracts JSON objects from token streams while handling nested quotes and currency symbols.
2. **Contract Validation**: Validates the payload against the strict JSON schema with zero-temperature retries on syntax deviations.
3. **Database Grounding**: Verifies cited record IDs against actual database tables, dropping hallucinated references before rendering.
4. **Deterministic Fallback**: If generation fails schema checks, returns a fact-derived structured response to maintain application stability.

In [ ]:
%%writefile /content/thalamus_runtime.py
"""
Serving runtime for THALAMUS -- Bangladesh edition.

The frontend streams this JSON straight into React components, so a parse error
is a broken page. Generation goes through extraction -> schema validation ->
grounding, with a low-temperature retry between, and a fact-derived answer as
the floor. The UI never sees an exception.

Identifier handling: the model is trained on snake_case agent ids (matching the
dataset YAML), validation runs on snake_case, and the wire format is converted
to kebab-case on the way out because that is what the Next.js frontend and the
routing YAML use. Inbound ids are accepted in either shape.
"""
from __future__ import annotations

import gc
import json
import re
import time
from typing import Any, Dict, List, Optional

import jsonschema

import thalamus_core as tc

# --------------------------------------------------------------------------
# The contract the frontend parses. additionalProperties is False on purpose so
# a chatty model cannot smuggle in a "notes" field the UI silently drops.
# The agent enum accepts both cases; coerce() normalises before validation runs.
# --------------------------------------------------------------------------
_AGENT_ENUM = sorted(set(tc.AGENTS) | {tc.to_kebab(a) for a in tc.AGENTS})

THALAMUS_SCHEMA = {
    "type": "object",
    "required": ["agent", "analyzed", "finding", "contributingFactors",
                 "evidence", "recommendedAction"],
    "additionalProperties": False,
    "properties": {
        "agent": {"type": "string", "enum": _AGENT_ENUM},
        "analyzed": {
            "type": "array", "minItems": 1,
            "items": {"type": "object", "required": ["domain", "records_scanned"],
                      "additionalProperties": False,
                      "properties": {"domain": {"type": "string"},
                                     "records_scanned": {"type": "integer", "minimum": 0}}},
        },
        "finding": {"type": "string", "minLength": 40},
        "contributingFactors": {
            "type": "array", "minItems": 1, "maxItems": 6,
            "items": {"type": "object", "required": ["factor", "magnitude"],
                      "additionalProperties": False,
                      "properties": {"factor": {"type": "string", "minLength": 3},
                                     "magnitude": {"type": "string"}}},
        },
        "evidence": {
            "type": "array", "minItems": 1,
            "items": {"type": "object", "required": ["table", "record_ids"],
                      "additionalProperties": False,
                      "properties": {"table": {"type": "string"},
                                     "record_ids": {"type": "array",
                                                    "items": {"type": "string"}}}},
        },
        "recommendedAction": {
            "type": "object",
            "required": ["action", "riskTier", "confidence", "requiresApproval"],
            "additionalProperties": False,
            "properties": {
                "action": {"type": "string", "minLength": 10},
                "riskTier": {"type": "string", "enum": ["low", "medium", "high"]},
                "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                "requiresApproval": {"type": "boolean"},
            },
        },
    },
}

_VALIDATOR = jsonschema.Draft7Validator(THALAMUS_SCHEMA)


# --------------------------------------------------------------------------
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    """Pull the first balanced JSON object out of a model completion.

    Handles the three things models actually do wrong: markdown fences, a prose
    preamble, and a trailing comma. Brace counting is string-aware so a `{` or
    `}` inside a finding does not end the object early -- which matters here
    because findings quote Bangla text and currency strings.
    """
    if not text:
        return None
    text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")

    start = text.find("{")
    if start == -1:
        return None

    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if esc:
            esc = False
            continue
        if ch == "\\":
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                blob = text[start:i + 1]
                try:
                    return json.loads(blob)
                except json.JSONDecodeError:
                    repaired = re.sub(r",\s*([}\]])", r"\1", blob)
                    try:
                        return json.loads(repaired)
                    except json.JSONDecodeError:
                        return None
    return None


def coerce(obj: Dict[str, Any], agent: str) -> Dict[str, Any]:
    """Nudge near-miss output into the contract before validating.

    Conservative by design: it fixes types, normalises identifiers and drops
    unknown keys, but never invents a missing `finding` or `evidence`. Those are
    real failures and should fall through to the fallback rather than be
    papered over with placeholder text.
    """
    if not isinstance(obj, dict):
        return obj

    obj["agent"] = tc.normalize_agent(obj.get("agent")) or agent

    ra = obj.get("recommendedAction")
    if isinstance(ra, dict):
        tier = str(ra.get("riskTier", "")).strip().lower()
        ra["riskTier"] = tier if tier in ("low", "medium", "high") else "medium"
        try:
            ra["confidence"] = max(0.0, min(1.0, float(ra.get("confidence", 0.7))))
        except (TypeError, ValueError):
            ra["confidence"] = 0.7
        approval = ra.get("requiresApproval")
        if isinstance(approval, str):
            approval = approval.strip().lower() in ("true", "yes", "1")
        ra["requiresApproval"] = bool(approval) if approval is not None else (ra["riskTier"] != "low")
        obj["recommendedAction"] = {k: ra[k] for k in
                                    ("action", "riskTier", "confidence", "requiresApproval")
                                    if k in ra}

    ev = obj.get("evidence")
    if isinstance(ev, list):
        obj["evidence"] = [{"table": str(e.get("table", "orders")),
                            "record_ids": [str(x) for x in (e.get("record_ids") or [])]}
                           for e in ev if isinstance(e, dict)]

    an = obj.get("analyzed")
    if isinstance(an, list):
        cleaned = []
        for a in an:
            if not isinstance(a, dict):
                continue
            try:
                cleaned.append({"domain": str(a.get("domain", "orders")),
                                "records_scanned": int(a.get("records_scanned", 0))})
            except (TypeError, ValueError):
                continue
        obj["analyzed"] = cleaned

    cf = obj.get("contributingFactors")
    if isinstance(cf, list):
        obj["contributingFactors"] = [
            {"factor": str(c.get("factor", "")), "magnitude": str(c.get("magnitude", ""))}
            for c in cf if isinstance(c, dict) and c.get("factor")][:6]

    return {k: v for k, v in obj.items() if k in THALAMUS_SCHEMA["properties"]}


def validate(obj: Dict[str, Any]) -> List[str]:
    return [f"{'/'.join(str(p) for p in e.path) or '<root>'}: {e.message}"
            for e in _VALIDATOR.iter_errors(obj)]


# order_items cite product ids; payments cite order ids. Map the citing table
# onto the table that actually owns that id universe.
_ID_UNIVERSE = {"order_items": "products", "payments": "orders", "policy": "suppliers"}


def check_grounding(obj: Dict[str, Any], whitelists: Dict[str, set]) -> Dict[str, Any]:
    """Verify every cited record_id exists; drop fabricated ones.

    An id that does not resolve is worse than no id: the UI renders it as a
    clickable record and the user hits a dead link, which destroys trust in
    every other number on the page.
    """
    bad, total = [], 0
    for block in obj.get("evidence", []):
        table = block.get("table", "")
        pool = (whitelists.get(table)
                or whitelists.get(_ID_UNIVERSE.get(table, ""))
                or set())
        kept = []
        for rid in block.get("record_ids", []):
            total += 1
            if not pool or rid in pool:
                kept.append(rid)
            else:
                bad.append(f"{table}:{rid}")
        block["record_ids"] = kept
    obj["evidence"] = [b for b in obj.get("evidence", []) if b.get("record_ids")]
    return {"grounded": not bad, "hallucinated_ids": bad, "checked": total}


def to_wire(obj: Dict[str, Any], kebab: bool = True) -> Dict[str, Any]:
    """Convert the validated object to the shape the frontend expects."""
    if kebab and obj.get("agent"):
        obj["agent"] = tc.to_kebab(tc.normalize_agent(obj["agent"]) or obj["agent"])
    return obj


# --------------------------------------------------------------------------
class Engine:
    def __init__(self, model, tokenizer, facts, tables, system_prompt,
                 max_new_tokens: int = 768, kebab_output: bool = True):
        self.model = model
        self.tok = tokenizer
        self.facts = facts
        self.tables = tables
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.kebab_output = kebab_output
        self.whitelists = tc.valid_ids(facts, tables)

    def build_prompt(self, query: str, agent: Optional[str],
                     scope: Optional[Dict] = None) -> str:
        agent = tc.normalize_agent(agent)
        domains = tc.route_domains(query, agent=agent)
        context = tc.render_context(self.facts, domains)
        user = query if not agent else f"[agent: {agent}] {query}"
        if scope:
            user += f"\n\nSCOPE: {json.dumps(scope, ensure_ascii=False)}"
        msgs = [{"role": "system", "content": self.system_prompt},
                {"role": "user", "content": f"{user}\n\nFACTS:\n{context}"}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def generate(self, prompt: str, temperature: float = 0.3) -> str:
        import torch
        inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=temperature,
                top_p=0.9,
                do_sample=temperature > 0,
                repetition_penalty=1.05,
                pad_token_id=self.tok.eos_token_id,
            )
        text = self.tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        del inputs, out
        return text

    def answer(self, query: str, agent: Optional[str] = None,
               scope: Optional[Dict] = None) -> Dict[str, Any]:
        t0 = time.time()
        requested = tc.normalize_agent(agent)
        resolved = requested or tc.infer_agent(query)
        prompt = self.build_prompt(query, requested, scope)

        for attempt, temp in enumerate((0.3, 0.0)):
            raw = self.generate(prompt, temperature=temp)
            obj = extract_json(raw)
            if obj is None:
                continue
            obj = coerce(obj, resolved)
            if validate(obj):
                continue
            g = check_grounding(obj, self.whitelists)
            if not obj.get("evidence"):
                continue
            obj = to_wire(obj, self.kebab_output)
            obj["_meta"] = {
                "status": "ok" if attempt == 0 else "repaired",
                "grounded": g["grounded"],
                "hallucinated_ids": g["hallucinated_ids"],
                "agent_internal": resolved,
                "latency_s": round(time.time() - t0, 2),
                "domains": tc.route_domains(query, agent=requested),
                "currency": tc.CURRENCY,
            }
            return obj

        obj = to_wire(self.fallback(query, resolved), self.kebab_output)
        obj["_meta"] = {"status": "fallback", "grounded": True, "hallucinated_ids": [],
                        "agent_internal": resolved,
                        "latency_s": round(time.time() - t0, 2),
                        "domains": tc.route_domains(query, agent=requested),
                        "currency": tc.CURRENCY}
        return obj

    def infer_agent(self, query: str) -> str:
        return tc.infer_agent(query)

    def fallback(self, query: str, agent: str) -> Dict[str, Any]:
        """Deterministic answer assembled from facts when generation fails.

        Reuses the label builders that produced the training data, so a fallback
        is indistinguishable in shape from a good generation.
        """
        import thalamus_dataset as td
        lang = "bn" if re.search(r"[\u0980-\u09FF]", query) else "en"
        agent = tc.normalize_agent(agent) or "sales_analyst"
        domains = tc.route_domains(query, agent=agent)
        preferred = {
            "revenue": "revenue_why", "inventory": "inventory_at_risk",
            "suppliers": "supplier_worst", "reviews": "reviews_sentiment",
            "customers": "customer_churn", "finance": "finance_margin",
            "policy": "policy_rules",
        }
        # automation_agent shares its domains with inventory, so a domain-only
        # lookup would hand back an inventory-branded answer for an automation
        # request. Pin the agent's own family first.
        agent_family = {"automation_agent": "automation_policy",
                        "policy_docs_agent": "policy_rules",
                        "marketing_agent": "customer_churn",
                        "customer_success": "reviews_sentiment"}
        fam = agent_family.get(agent)
        if fam:
            label = td.build_label(fam, self.facts, lang)
            if label:
                return label
        # Try the agent's own domains first so a policy question does not fall
        # back to a sales answer.
        ordered = tc.AGENT_DOMAINS.get(agent, []) + domains + ["revenue"]
        for d in ordered:
            fam = preferred.get(d)
            if not fam:
                continue
            label = td.build_label(fam, self.facts, lang)
            if label:
                return label
        return {
            "agent": agent,
            "analyzed": [{"domain": "orders", "records_scanned": 0}],
            "finding": ("এই প্রশ্নের উত্তর দেওয়ার মতো পর্যাপ্ত হিসাবকৃত তথ্য নেই। অনুগ্রহ করে "
                        "একটি নির্দিষ্ট পণ্য, সাপ্লায়ার বা মাস উল্লেখ করে আবার জিজ্ঞাসা করুন।"
                        if lang == "bn" else
                        "There is not enough computed evidence in the connected data to answer this "
                        "reliably. Try narrowing it to a specific product, supplier or month."),
            "contributingFactors": [{"factor": "Insufficient evidence in scope", "magnitude": "n/a"}],
            "evidence": [{"table": "orders", "record_ids": []}],
            "recommendedAction": {
                "action": "Rephrase with a specific SKU, supplier or time window",
                "riskTier": "low", "confidence": 0.3, "requiresApproval": False},
        }


def free_memory():
    """Release cached CUDA blocks. Call before loading PEFT and before inference."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass

In [ ]:
import thalamus_runtime as trt
importlib.reload(trt)

trt.free_memory()                       # clear training allocations before generation
FastLanguageModel.for_inference(model)  # ~2x faster decode

engine = trt.Engine(model=model, tokenizer=tokenizer, facts=FACTS, tables=tables,
                    system_prompt=td.SYSTEM_PROMPT, kebab_output=True)

TESTS = [
    ("root query", "Why did revenue decline last month?", None),
    ("inventory ", "List products at risk of stockout with reorder recommendations.", "inventory-agent"),
    ("bangla    ", "কোন সাপ্লায়ারের ডেলিভারি সবচেয়ে বেশি দেরি হচ্ছে?", None),
    ("policy    ", "Does this restock need my approval?", "policy-docs-agent"),
]

for tag, q, agent in TESTS:
    r = engine.answer(q, agent=agent)
    m = r["_meta"]
    print(f"\n{'='*78}\n{tag} | {q}")
    print(f"status={m['status']}  agent={r['agent']}  grounded={m['grounded']}  "
          f"latency={m['latency_s']}s")
    print(json.dumps({k: v for k, v in r.items() if k != "_meta"},
                     ensure_ascii=False, indent=2)[:1700])

In [ ]:
# Batch evaluation over the held-out split.
trt.free_memory()
N = min(40, len(val_raw))
schema_ok = grounded_ok = agent_ok = lang_ok = 0

for i, r in enumerate(val_raw[:N]):
    out = engine.answer(r["instruction"], agent=r["agent"])
    m = out["_meta"]
    schema_ok   += m["status"] in ("ok", "repaired")
    grounded_ok += m["grounded"]
    agent_ok    += tc.normalize_agent(out.get("agent")) == r["agent"]
    has_bn = bool(re.search(r"[\u0980-\u09FF]", out.get("finding", "")))
    lang_ok += (has_bn if r["lang"] == "bn" else not has_bn)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N}")

print(f"\n{'metric':<30} {'rate':>8}")
print(f"{'valid JSON (post-repair)':<30} {100*schema_ok/N:>7.1f}%")
print(f"{'all record_ids grounded':<30} {100*grounded_ok/N:>7.1f}%")
print(f"{'correct agent routing':<30} {100*agent_ok/N:>7.1f}%")
print(f"{'answered in query language':<30} {100*lang_ok/N:>7.1f}%")
print("\nBelow ~95% grounding: raise epochs to 3 or widen context_budget.")
print("Frequent 'repaired' status: undertrained, or the response mask is wrong.")

## 6. Real-Time Serving Layer (FastAPI + Tunneling)

Exposes the intelligence engine via REST endpoints compatible with the Next.js workspace frontend. Both snake_case and kebab-case agent identifiers are accepted and mapped to uniform kebab-case responses.

In [ ]:
from google.colab import userdata
from pyngrok import ngrok, conf

# Key icon in the left sidebar: Secrets > NGROK_AUTH_TOKEN
try:
    conf.get_default().auth_token = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    print("Set NGROK_AUTH_TOKEN in Colab Secrets (free at dashboard.ngrok.com).")

In [ ]:
import nest_asyncio, uvicorn, threading, time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any

nest_asyncio.apply()
app = FastAPI(title="THALAMUS Intelligence Engine", version="2.0-BD")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*", "http://localhost:3000", "http://127.0.0.1:3000"],
    allow_credentials=False,     # cannot be True alongside a "*" origin
    allow_methods=["*"], allow_headers=["*"],
)

class AskRequest(BaseModel):
    query: str = Field(..., min_length=2, max_length=1000)
    agent: Optional[str] = None

class AgentRequest(BaseModel):
    task: str = Field(..., min_length=2, max_length=1000)
    scope: Dict[str, Any] = Field(default_factory=dict)

@app.get("/health")
def health():
    return {"status": "ok", "model": CFG["model_name"], "currency": tc.CURRENCY,
            "business": FACTS.get("policy", {}).get("business_name"),
            "anchor_month": str(FACTS.get("_anchor_month")),
            "agents": [tc.to_kebab(a) for a in tc.AGENTS]}

@app.get("/api/agents")
def list_agents():
    return [{"id": tc.to_kebab(a), "internal_id": a, "name": tc.AGENT_LABELS[a],
             "domains": tc.AGENT_DOMAINS[a],
             "defaultRiskTier": td.RISK_TIER[a]} for a in tc.AGENTS]

@app.get("/api/facts")
def get_facts(domain: Optional[str] = None):
    """Backs the Business Brain graph and lets you see what the model saw."""
    if domain:
        if domain not in FACTS:
            raise HTTPException(404, f"unknown domain: {domain}")
        return FACTS[domain]
    return {k: v for k, v in FACTS.items() if not k.startswith("_")}

@app.post("/api/ask")
def ask(req: AskRequest):
    agent = tc.normalize_agent(req.agent)
    if req.agent and not agent:
        raise HTTPException(422, f"unknown agent: {req.agent}")
    return engine.answer(req.query, agent=agent)

@app.post("/api/agents/{agent_id}/run")
def run_agent(agent_id: str, req: AgentRequest):
    agent = tc.normalize_agent(agent_id)
    if not agent:
        raise HTTPException(404, {"error": f"unknown agent: {agent_id}",
                                  "valid": [tc.to_kebab(a) for a in tc.AGENTS]})
    return engine.answer(req.task, agent=agent, scope=req.scope)

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True).start()
time.sleep(4)

ngrok.kill()
PUBLIC_URL = str(ngrok.connect(8000).public_url)
print("public URL:", PUBLIC_URL)

In [ ]:
import requests

print(requests.get(f"{PUBLIC_URL}/health", timeout=30).json(), "\n")

# Both identifier shapes must resolve.
for aid in ["inventory-agent", "inventory_agent", "policy-docs-agent"]:
    r = requests.post(f"{PUBLIC_URL}/api/agents/{aid}/run",
                      json={"task": "What needs attention?", "scope": {"limit": 5}}, timeout=180)
    print(f"POST /api/agents/{aid}/run -> {r.status_code} "
          f"agent={r.json().get('agent') if r.ok else r.text[:80]}")

r = requests.post(f"{PUBLIC_URL}/api/ask",
                  json={"query": "গত মাসে বিক্রি কমল কেন?"}, timeout=180)
print(f"\nPOST /api/ask (bangla) -> {r.status_code}")
print(json.dumps(r.json(), ensure_ascii=False, indent=2)[:1200])

In [ ]:
print(f"""
Paste into .env.local at the root of frontend_thalamusAI, then restart the dev server.

  THALAMUS_LLM_PROVIDER=colab
  THALAMUS_LLM_BASE_URL={PUBLIC_URL}
  THALAMUS_ASK_ENDPOINT={PUBLIC_URL}/api/ask
  THALAMUS_AGENT_ENDPOINT={PUBLIC_URL}/api/agents
  THALAMUS_REQUEST_TIMEOUT_MS=120000
  THALAMUS_CURRENCY=BDT

  # Server-only. Do NOT prefix with NEXT_PUBLIC_ -- the tunnel has no auth, and
  # a public URL in the browser bundle is an open endpoint on your GPU.
""")

### 6.1 Frontend Gateway Proxy (`src/app/api/ask/route.ts`)

A Next.js server-side route handler that proxies client queries to the Python inference engine, providing secure network isolation, timeout handling, and payload validation.

In [ ]:
print(r'''
// src/app/api/ask/route.ts
import { NextRequest, NextResponse } from "next/server";

export const runtime = "nodejs";
export const maxDuration = 120;

const AGENTS = [
  "sales-analyst",
  "marketing-agent",
  "inventory-agent",
  "customer-success",
  "finance-agent",
  "policy-docs-agent",
  "automation-agent",
] as const;

type AgentId = (typeof AGENTS)[number];

/** Accept snake_case, kebab-case or display names; emit kebab-case. */
function normalizeAgent(input?: string | null): AgentId | null {
  if (!input) return null;
  const kebab = input.trim().toLowerCase().replace(/[\s_]+/g, "-");
  return (AGENTS as readonly string[]).includes(kebab) ? (kebab as AgentId) : null;
}

export interface ThalamusInsight {
  agent: AgentId;
  analyzed: { domain: string; records_scanned: number }[];
  finding: string;
  contributingFactors: { factor: string; magnitude: string }[];
  evidence: { table: string; record_ids: string[] }[];
  recommendedAction: {
    action: string;
    riskTier: "low" | "medium" | "high";
    confidence: number;
    requiresApproval: boolean;
  };
}

export async function POST(req: NextRequest) {
  let body: { query?: string; agent?: string };
  try {
    body = await req.json();
  } catch {
    return NextResponse.json({ error: "invalid JSON body" }, { status: 400 });
  }

  const query = body.query?.trim();
  if (!query) {
    return NextResponse.json({ error: "query is required" }, { status: 400 });
  }

  // An unrecognised agent is a client bug worth surfacing, not something to
  // silently drop -- otherwise the request quietly runs as the wrong agent.
  const agent = normalizeAgent(body.agent);
  if (body.agent && !agent) {
    return NextResponse.json(
      { error: `unknown agent: ${body.agent}`, valid: AGENTS },
      { status: 422 }
    );
  }

  const endpoint = process.env.THALAMUS_ASK_ENDPOINT;
  if (!endpoint) {
    return NextResponse.json(
      { error: "engine_not_configured" },
      { status: 503 }
    );
  }

  const controller = new AbortController();
  const timeout = setTimeout(
    () => controller.abort(),
    Number(process.env.THALAMUS_REQUEST_TIMEOUT_MS ?? 120_000)
  );

  try {
    const res = await fetch(endpoint, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ query, agent }),
      signal: controller.signal,
      cache: "no-store",
    });

    if (!res.ok) throw new Error(`upstream ${res.status}`);

    const data = await res.json();
    const { _meta, ...insight } = data as ThalamusInsight & { _meta?: unknown };

    if (!insight.finding || !insight.recommendedAction || !Array.isArray(insight.evidence)) {
      throw new Error("malformed insight from engine");
    }

    return NextResponse.json(insight, {
      headers: {
        "x-thalamus-status": (_meta as any)?.status ?? "unknown",
        "x-thalamus-grounded": String((_meta as any)?.grounded ?? ""),
      },
    });
  } catch (err) {
    console.error("[thalamus/ask]", err);
    const aborted = err instanceof Error && err.name === "AbortError";
    return NextResponse.json(
      { error: aborted ? "engine_timeout" : "engine_unavailable" },
      { status: aborted ? 504 : 502 }
    );
  } finally {
    clearTimeout(timeout);
  }
}
''')

## 7. Business Analytics & Strategic Findings

Empirical audit of the underlying SME dataset demonstrates how the intelligence engine interprets core business drivers:

### 1. Revenue Stability vs. SKU Churn
* **Top-Line Stability**: Month-over-month revenue changed by −0.80% (৳5,264,105 vs. ৳5,306,644), sitting within normal operating variance.
* **Internal SKU Mix Shifts**: While overall revenue remained flat, significant product-level reallocation occurred: six products gained ~৳60k–৳90k each while six others dropped by equivalent volumes, highlighting changing category preferences.

### 2. Fulfillment Metrics & Customer Sentiment
* **Fulfillment Baseline**: Supplier late-delivery rates average 7.5% fleet-wide with tight vendor consistency (7.1% to 8.1%).
* **Sentiment Independence**: Review score does not correlate with delivery delays (r = −0.007; late orders average 3.57 stars vs. 3.60 for on-time orders). Negative reviews stem from broader product experience rather than logistics delays.

### 3. High-Impact Operational Priorities
* **Inventory Stockout Risk**: 24 out of 100 SKUs require replenishment, with total reorder commitments of ৳10.8M requiring staged purchase orders.
* **Vendor Contract Adherence**: Chittagong Trade House (SUP002) shows a measurable SLA gap (98% contracted reliability vs. 91.5% actual fulfillment).
* **Payment Methods & Cash Flow**: 65% of volume processes through digital mobile wallets (bKash/Nagad) while 20% relies on Cash-on-Delivery, directly linking fulfillment success with working capital velocity.